# 06 — PM2.5 รายตำบลจากข้อมูลดาวเทียม และการตรวจสอบกับสถานีภาคพื้น

**Part 1 · การกระจายของ PM2.5 → ค่าเฉลี่ยรายตำบล** — ระดับตำบลมีสถานีเพียง 2 แห่ง จึงใช้ PM2.5 ผิวพื้นที่ประมาณจากดาวเทียมเป็นตัวแปรการสัมผัสระดับตำบล
(เหตุผลที่ไม่สร้างแบบจำลองเองจาก AOD ดู Notebook 05: spatial-block CV)

**ชุดข้อมูล:** SatPM2.5 **V6.GL.03** (Atmospheric Composition Analysis Group, Washington University in St. Louis; ต่อยอดจาก van Donkelaar et al., 2021)
ค่าเฉลี่ยรายเดือน 0.01° × 0.01° (~1.1 km) ปี 1998–2024 รวม AOD จากดาวเทียมหลายดวง แบบจำลอง GEOS-Chem และสถานีภาคพื้นทั่วโลก
เผยแพร่แบบสาธารณะใน AWS Open Data (`s3://satpmdata`) — notebook อ่าน **เฉพาะกรอบพื้นที่รอบสระบุรี** แล้วเก็บ cache ลง Drive

| ขั้น | เนื้อหา |
|---|---|
| 1 | ขอบเขตตำบล (OCHA COD-AB → GADM) และประชากร WorldPop 2020 ความละเอียด 100 m |
| 2 | ดึง PM2.5 ดาวเทียมรายเดือน 2016–2024 เฉพาะกรอบพื้นที่ |
| 3 | Zonal statistics 2 แบบ: **ถ่วงพื้นที่ซ้อนทับ** และ **ถ่วงประชากร** (ตัวแปรการสัมผัสหลักในโครงร่าง) |
| 4 | ตรวจสอบกับสถานี คพ. ทุกสถานีในรัศมี 200 km (r, bias, NMB, NME, RMSE) แยกฤดู · เน้น 24T/25T |
| 5 | ทักษะเชิงพื้นที่: ดาวเทียมจัดอันดับ "ที่ไหนสูงกว่า" ได้ถูกหรือไม่ (Spearman ระหว่างสถานี) |
| 6 | แผนที่ค่าเฉลี่ยรายตำบล: ภาพรวม · รายฤดู · รายปี · ควอร์ไทล์ |
| 7 | **ตาราง PM2.5 รายตำบลรายเดือน ต.ค. 2564 – ก.ย. 2569** สำหรับเชื่อมกับข้อมูลผู้ป่วย HDC รายเดือน (ม.ค. 2568 เป็นต้นไปใช้สมการที่ 15) + ทดสอบย้อนหลัง + ค่ารายปีงบประมาณ |
| 8 | บันทึกไฟล์ (Notebook 07 ใช้ต่อ) และข้อความในบทวิธีวิจัย |

> `Runtime → Run all` ครั้งแรก ~10–30 นาที (ดึงข้อมูลดาวเทียม 108 เดือน) ครั้งต่อไปเร็วขึ้นมากเพราะใช้ cache ใน Drive

## 0. CONFIG

In [ ]:
PROVINCE_EN = "Saraburi"
BOUNDARY_SOURCE = "drive"       # "drive" (shapefile ตำบลใน Google Drive) → สำรอง "cod" → "gadm" → "osm" อัตโนมัติ
TAMBON_SHP = ""                 # path ไฟล์ขอบเขตตำบลใน Drive เช่น "GIS/tambon_saraburi.shp" (.shp/.gpkg/.zip/.geojson) — เว้นว่าง = ค้นหาอัตโนมัติ
SAT_S3 = "https://satpmdata.s3.amazonaws.com"
SAT_VERSION = "V6GL03"                          # SatPM2.5 V6.GL.03 (1998–2024)
SAT_PRODUCT = "CNNPM25"                         # โฟลเดอร์ 0.01° ใน bucket (CNNPM25c0p10 = 0.1°)
SAT_REGIONS = ["AS", "GL"]                      # ลองไฟล์ภูมิภาคเอเชียก่อน (เล็กกว่า) แล้วจึงไฟล์ทั่วโลก
SAT_TEMPORAL = "Monthly"                        # "Monthly" (หลัก) หรือ "Annual" (เร็วกว่า ใช้ทดลอง)
SAT_START, SAT_END = "2016-01", "2024-12"       # ช่วงที่ซ้อนกับข้อมูลสถานี (ชุดข้อมูลมีถึง 2024)
SAT_LOCAL_DIR = ""                              # (ไม่บังคับ) โฟลเดอร์ใน Drive ที่มีไฟล์ .nc ที่ดาวน์โหลดเอง — เว้นว่าง = ดึงจาก S3
SAT_CACHE_DIR = "/content/drive/MyDrive/3-ColabProjects/Training/SatPM25_cache"
PCD_FOLDER_ID = "1pXbM_yiwL3PqMsEq8QbPJWqC2Z71DKSe"   # โฟลเดอร์ Google Drive "PCD_PM25" (2016.xlsx … 2026.xlsx ของ คพ.)
PCD_PATH = ""
QC_FILE = "pm25_daily_qc_24T_25T.csv"           # 24T/25T ที่ผ่าน QC จาก Notebook 01 (ถ้ามี)
CENTER = (14.53, 100.91)
VALID_RADIUS_KM = 200                           # สถานี คพ. ที่ใช้ตรวจสอบ
MIN_PCT = 75
WORLDPOP_100M = True                            # False = ใช้ไฟล์ 1 km (เร็วกว่า หยาบกว่า)
WP_CACHE_DIR = "/content/drive/MyDrive/3-ColabProjects/Training/WorldPop_cache"   # เก็บ WorldPop เฉพาะกรอบจังหวัด (ใช้ร่วมกับ 08)
SAVE_TO_DRIVE = True                            # Notebook 07 อ่านผลจาก Drive

In [ ]:
import importlib.util, subprocess, sys
need = {"rasterio": "rasterio", "xarray": "xarray", "h5netcdf": "h5netcdf", "netCDF4": "netCDF4", "fsspec": "fsspec",
        "aiohttp": "aiohttp", "openpyxl": "openpyxl"}
missing = [p for m, p in need.items() if importlib.util.find_spec(m) is None]
if missing:
    print("ติดตั้ง:", missing); subprocess.run(f"{sys.executable} -m pip -q install " + " ".join(missing), shell=True)

In [ ]:
import os, io, sys, json, math, time, zipfile, warnings, subprocess
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

UA = {"User-Agent": "Mozilla/5.0 (academic research notebook; PM2.5 Saraburi)"}

def setup_thai_font():
    """ให้กราฟ matplotlib แสดงภาษาไทยได้ (ฟอนต์เริ่มต้นไม่มีตัวอักษรไทย → แสดงเป็น □□□)"""
    import matplotlib.font_manager as fm, glob
    base = "/content" if os.path.isdir("/content") else "."
    path = os.path.join(base, "Sarabun-Regular.ttf")
    if not os.path.exists(path):
        try:
            r = requests.get("https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf", timeout=60)
            if r.status_code == 200 and len(r.content) > 50000: open(path, "wb").write(r.content)
        except Exception:
            pass
    if not os.path.exists(path):                                  # สำรอง: ฟอนต์ไทยจาก apt (TLWG)
        subprocess.run("apt-get -qq install -y fonts-thai-tlwg > /dev/null 2>&1", shell=True)
        cand = sorted(glob.glob("/usr/share/fonts/**/tlwg/Loma.*tf", recursive=True)) + \
               sorted(glob.glob("/usr/share/fonts/**/Garuda.*tf", recursive=True))
        path = cand[0] if cand else None
    if path and os.path.exists(path):
        fm.fontManager.addfont(path)
        name = fm.FontProperties(fname=path).get_name()
        plt.rcParams["font.family"] = ["DejaVu Sans", name]      # ละติน = DejaVu, ไทย = fallback ไปฟอนต์ไทย
        print(f"✓ ฟอนต์ไทยสำหรับกราฟ: {name}")
    else:
        print("⚠ ติดตั้งฟอนต์ไทยไม่ได้ — ข้อความภาษาไทยในกราฟจะแสดงเป็นสี่เหลี่ยม")
setup_thai_font()

# เว็บราชการบางแห่ง (เช่น Air4Thai) ส่งใบรับรอง SSL ไม่ครบ chain → Python ตรวจสอบไม่ผ่าน (SSLError)
# อนุญาตให้ข้ามการตรวจใบรับรองเฉพาะโดเมนเหล่านี้ (ข้อมูลสาธารณะ อ่านอย่างเดียว)
SSL_RELAX_HOSTS = ("air4thai.pcd.go.th", "air4thai.com", "www.air4thai.com")

def http_get(url, timeout=(15, 120), tries=3, quiet=False, **kw):
    """GET พร้อม retry — คืน Response เมื่อ HTTP 200 ไม่เช่นนั้น raise"""
    last = None
    host = url.split("/")[2] if "://" in url else ""
    verify = kw.pop("verify", True)
    for k in range(tries):
        try:
            try:
                r = requests.get(url, headers=UA, timeout=timeout, verify=verify, **kw)
            except requests.exceptions.SSLError:
                if host not in SSL_RELAX_HOSTS or verify is False:
                    raise
                import urllib3; urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
                if not quiet: print(f"   (ใบรับรอง SSL ของ {host} ไม่สมบูรณ์ — เชื่อมต่อโดยไม่ตรวจใบรับรอง)")
                verify = False
                r = requests.get(url, headers=UA, timeout=timeout, verify=False, **kw)
            if r.status_code == 200:
                return r
            last = f"HTTP {r.status_code}"
            if r.status_code == 429 and k < tries - 1:     # ขอถี่เกินโควตา (เช่น Open-Meteo) → รอนานขึ้นแล้วลองใหม่
                wait = int(r.headers.get("Retry-After", 0) or 0) or 30 * (k + 1)
                print(f"   HTTP 429 จาก {host} — รอ {wait} s แล้วลองใหม่", flush=True)
                time.sleep(wait); continue
        except Exception as e:
            last = type(e).__name__ + ": " + str(e)[:100]
        if k < tries - 1:
            time.sleep(2 * (k + 1))
    raise RuntimeError(f"{last}  <- {url[:110]}")

def month_chunks(start, end):
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    out, cur = [], s
    while cur <= e:
        nxt = min(cur + pd.offsets.MonthEnd(0), e)
        out.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt + pd.Timedelta(days=1)
    return out

from scipy import stats, sparse
import shapely
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [ ]:
# ---- ที่เก็บผลลัพธ์ (ถ้า SAVE_TO_DRIVE=True จะเก็บใน Google Drive เพื่อให้ Notebook อื่นใช้ต่อได้) ----
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTDIR = "/content/drive/MyDrive/PM25_Saraburi_notebooks"
    except Exception as e:
        print("⚠ mount Drive ไม่ได้ จะเก็บไว้ใน /content แทน:", e)
        OUTDIR = "/content/PM25_Saraburi_notebooks"
else:
    OUTDIR = "/content/PM25_Saraburi_notebooks" if os.path.isdir("/content") else "./PM25_Saraburi_notebooks"
os.makedirs(OUTDIR, exist_ok=True)
print("OUTDIR =", OUTDIR)
SAT_CACHE = SAT_CACHE_DIR if os.path.isdir("/content/drive/MyDrive") else os.path.join(OUTDIR, "SatPM25_cache")
os.makedirs(SAT_CACHE, exist_ok=True)
print("SAT_CACHE =", SAT_CACHE)

In [ ]:
# =====================================================================
#  เตรียมข้อมูลระดับตำบล (ดึงจากอินเทอร์เน็ตทั้งหมด)
#  1) ขอบเขตตำบล  : shapefile ใน Google Drive → OCHA COD-AB Thailand (มีรหัสตำบล 6 หลักตรงกับ HDC) → สำรอง GADM 4.1 → OpenStreetMap
#  2) ประชากร      : WorldPop 2020 (1 km, ปรับรวมรายตำบลด้วย zonal sum)
# =====================================================================
import geopandas as gpd
from shapely.geometry import Point
UTM = "EPSG:32647"

COD_BASE = "https://codgis.itos.uga.edu/arcgis/rest/services/COD_External/THA_pcode/FeatureServer"
GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_THA_3.json.zip"

def load_tambons_cod():
    """OCHA COD-AB Thailand (ที่มา: กรมแผนที่ทหาร/RTSD) ผ่าน live service ของ ITOS — มีรหัสตำบล (pcode) และชื่อภาษาไทย"""
    import re
    from urllib.parse import quote
    meta = http_get(COD_BASE + "?f=json", timeout=(15, 60)).json()
    layers = meta.get("layers", [])
    lyr = next((l for l in layers if re.search(r"(adm(in)?\s*_?3|tambon|sub.?district)", l["name"], re.I)), None)
    if lyr is None:
        raise RuntimeError(f"ไม่พบชั้นข้อมูลระดับ 3: {[l['name'] for l in layers]}")
    for where in [f"ADM1_EN = '{PROVINCE_EN}'", "ADM1_PCODE = 'TH19'", f"ADM1_EN LIKE '%{PROVINCE_EN}%'"]:
        url = f"{COD_BASE}/{lyr['id']}/query?where={quote(where)}&outFields=*&outSR=4326&f=geojson"
        try:
            g = gpd.read_file(io.BytesIO(http_get(url, timeout=(15, 300), tries=2).content))
        except Exception:
            continue
        if len(g):
            break
    else:
        raise RuntimeError("query ไม่ได้ข้อมูล")
    g = g.set_crs("EPSG:4326", allow_override=True)
    namecol = next(c for c in ["ADM3_TH", "ADM3_EN", "admin3Name_th", "admin3Name_en"] if c in g)
    pc = next(c for c in ["ADM3_PCODE", "admin3Pcode"] if c in g)
    g["tambon_id"] = g[pc]; g["name"] = g[namecol]
    g["a_code"] = g[pc].astype(str).str.replace(r"\D", "", regex=True).str[:6]     # TH190101 → 190101 (ตรงกับ HDC)
    if "ADM3_EN" in g: g["name_en"] = g["ADM3_EN"]
    return g[["tambon_id", "a_code", "name"] + (["name_en"] if "name_en" in g else []) + ["geometry"]], \
           f"OCHA COD-AB Thailand (ITOS live service, layer '{lyr['name']}')"

def load_tambons_gadm():
    """GADM 4.1 ระดับ 3 (ตำบล) — ไฟล์ทั้งประเทศ ~30–60 MB"""
    print("   ดาวน์โหลด GADM 4.1 THA level 3 ...")
    raw = http_get(GADM_URL, timeout=(15, 900)).content
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        name = next(n for n in z.namelist() if n.endswith(".json"))
        g = gpd.read_file(io.BytesIO(z.read(name)))
    g = g[g["NAME_1"].str.replace(" ", "").str.contains(PROVINCE_EN, case=False, na=False)].copy()
    g["tambon_id"] = g["GID_3"]
    g["name"] = g["NL_NAME_3"].where(g.get("NL_NAME_3", pd.Series(dtype=str)).notna() & (g.get("NL_NAME_3", "") != "NA"), g["NAME_3"]) \
        if "NL_NAME_3" in g else g["NAME_3"]
    g["name_en"] = g["NAME_3"]
    cols = ["tambon_id", "name", "name_en", "geometry"]
    if "CC_3" in g and g["CC_3"].astype(str).str.fullmatch(r"\d{6}").mean() > 0.8:
        g["a_code"] = g["CC_3"].astype(str); cols.insert(1, "a_code")
    return g[cols].to_crs("EPSG:4326"), "GADM 4.1 (THA level 3)"

def load_tambons_osm():
    import osmnx as ox
    ox.settings.timeout = 600
    prov = ox.geocode_to_gdf(f"{PROVINCE_EN} Province, Thailand")
    # หมายเหตุ: osmnx รวมแท็กแบบ OR → ถ้าใส่ทั้ง boundary และ admin_level จะได้ขอบเขตทุกระดับ (จังหวัด/อำเภอ/ตำบล) ปนกัน
    g = ox.features_from_polygon(prov.geometry.iloc[0], {"admin_level": "8"}).reset_index()
    g = g[g.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    if "boundary" in g: g = g[g["boundary"].astype(str) == "administrative"]
    if "admin_level" in g: g = g[g["admin_level"].astype(str) == "8"]
    if "element" in g and (g["element"] == "relation").any(): g = g[g["element"] == "relation"]
    g = g[g.representative_point().within(prov.geometry.iloc[0])]
    namecol = next(c for c in ["name:th", "name", "name:en"] if c in g)
    g["tambon_id"] = g.get("id", g.index).astype(str); g["name"] = g[namecol]
    if "name:en" in g: g["name_en"] = g["name:en"]
    return g[["tambon_id", "name", "geometry"] + (["name_en"] if "name_en" in g else [])], "OpenStreetMap admin_level=8"

# ---------- ขอบเขตตำบลจาก shapefile ใน Google Drive ----------
TAMBON_SEARCH_DIRS = ["/content/drive/MyDrive/PM25_Saraburi_notebooks/boundary",   # ที่แนะนำ
                      "/content/drive/MyDrive/PM25_Saraburi_notebooks",
                      "/content/drive/MyDrive"]                                    # ค้นหาลึกสุด 4 ชั้น
TAMBON_NAME_PAT = r"(tambon|sub.?district|subdist|adm3|admin3|ตำบล|tb_?|_tam)"
SARA_BBOX = (100.55, 14.25, 101.40, 15.05)
MISSING_PARTS = []
TAMBON_CACHE = "saraburi_tambon_boundary.gpkg"   # สำเนาเฉพาะสระบุรี (เล็ก อ่านเร็ว) สร้างอัตโนมัติข้างไฟล์ต้นฉบับ

def find_tambon_files():
    """คืนรายการ path ของ shapefile/gpkg/geojson/zip ที่อาจเป็นขอบเขตตำบลใน Drive (TAMBON_SHP ก่อน แล้วค้นหาอัตโนมัติ)"""
    import re, glob
    MISSING_PARTS.clear()
    if not os.path.isdir("/content/drive/MyDrive"):
        try:
            from google.colab import drive; drive.mount("/content/drive")
        except Exception as e:
            raise RuntimeError(f"mount Drive ไม่ได้: {str(e)[:80]}")
    p = (globals().get("TAMBON_SHP") or "").strip()
    if p:
        if not p.startswith("/"): p = os.path.join("/content/drive/MyDrive", p)
        if os.path.exists(p): return [p]
        raise RuntimeError(f"ไม่พบไฟล์ TAMBON_SHP = {p}")
    exts = (".shp", ".gpkg", ".geojson", ".json", ".zip"); found = []
    for root0 in TAMBON_SEARCH_DIRS:
        if not os.path.isdir(root0): continue
        hits, base_depth = [], root0.rstrip("/").count("/")
        for root, dirs, files in os.walk(root0):
            if root.count("/") - base_depth >= 4: dirs[:] = []
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("SatPM25_cache", "ERA5_cache", "WorldPop_cache", "GDAS", "PCD_PM25")]
            for fn in files:
                if fn.lower().endswith(".shx") and re.search(TAMBON_NAME_PAT, fn, re.I) and \
                        (fn[:-4] + ".shp").lower() not in {f.lower() for f in files}:
                    MISSING_PARTS.append(os.path.join(root, fn[:-4] + ".shp"))         # มี .shx แต่ไม่มี .shp
                anyname = root.rstrip("/").endswith("/boundary")          # ในโฟลเดอร์ boundary รับทุกชื่อไฟล์
                if fn.lower().endswith(exts) and (anyname or re.search(TAMBON_NAME_PAT, fn, re.I)) \
                        and not re.search(r"(rates|gistar|lisa|roadside|summary|result)", fn, re.I):
                    if fn.lower().endswith(".shp"):
                        stem = fn[:-4]; have = {f.lower() for f in files}
                        if not {stem.lower() + ".dbf", stem.lower() + ".shx"} <= have:
                            MISSING_PARTS.append(os.path.join(root, fn)); continue      # .dbf/.shx ยังไม่ครบ (อัปโหลดไม่เสร็จ?)
                    hits.append(os.path.join(root, fn))
            if root0 == "/content/drive/MyDrive" and len(hits) > 50: break
        if hits:
            order = {".shp": 0, ".gpkg": 1, ".zip": 2, ".geojson": 3, ".json": 4}
            rank = lambda h: 0 if re.search(TAMBON_NAME_PAT, os.path.basename(h), re.I) else \
                             2 if re.search(r"(adm[0-2]|admin[0-2]|amphoe|province|district|_ln|_pt|line|point)", os.path.basename(h), re.I) else 1
            hits.sort(key=lambda h: (rank(h), order[os.path.splitext(h)[1].lower()], len(h)))
            found += [h for h in hits if h not in found]
            if any(os.path.basename(h) == TAMBON_CACHE for h in found): break      # เจอสำเนาแล้ว ไม่ต้องค้นต่อ
    found.sort(key=lambda h: os.path.basename(h) != TAMBON_CACHE)                # สำเนาเฉพาะสระบุรีมาก่อน
    if found: return found
    extra = f" (พบ .shp ที่ไฟล์ประกอบไม่ครบ: {[os.path.basename(m) for m in MISSING_PARTS[:3]]})" if MISSING_PARTS else ""
    raise RuntimeError("ไม่พบ shapefile ตำบลใน Drive" + extra + " — วางไฟล์ไว้ที่ MyDrive/PM25_Saraburi_notebooks/boundary/ "
                       "(ชื่อไฟล์มีคำว่า tambon/subdistrict/ตำบล) หรือกำหนด TAMBON_SHP")

def read_tambon_file(path):
    """อ่านไฟล์ขอบเขต + แก้ภาษาไทยเพี้ยน (TIS-620) + กำหนด CRS + ตัดเฉพาะสระบุรี + หาคอลัมน์รหัส/ชื่อตำบล"""
    import re
    src, layer = path, None
    if path.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            inner = [n for n in z.namelist() if n.lower().endswith((".shp", ".gpkg", ".geojson", ".json"))]
        if not inner: raise RuntimeError(f"ใน {os.path.basename(path)} ไม่มี .shp/.gpkg/.geojson")
        lvl = lambda n: (0 if re.search(r"(adm3|admin3|tambon|sub.?dist|ตำบล)", n, re.I) else
                         2 if re.search(r"(adm[0-2]|admin[0-2]|amphoe|province|district|line|pt|point)", n, re.I) else 1)
        inner.sort(key=lambda n: (lvl(n), len(n)))
        src = f"zip://{path}!{inner[0]}"
    if src.lower().endswith(".gpkg"):
        try:
            import pyogrio
            names = [l[0] for l in pyogrio.list_layers(src)]
        except Exception:
            names = []
        if len(names) > 1:
            pref = [n for n in names if re.search(r"(adm3|admin3|tambon|sub.?dist|ตำบล)", n, re.I)]
            layer = pref[0] if pref else names[-1]
    kw = {"layer": layer} if layer else {}
    g = None
    if os.path.getsize(path) > 50e6:                    # ไฟล์ทั้งประเทศขนาดใหญ่ → อ่านเฉพาะกรอบสระบุรี (เร็วกว่ามาก)
        try:
            g = gpd.read_file(src, bbox=(SARA_BBOX[0] - .3, SARA_BBOX[1] - .3, SARA_BBOX[2] + .3, SARA_BBOX[3] + .3), **kw)
            if len(g) < MIN_TAMBONS: g = None           # พิกัดในไฟล์ไม่ใช่ lat/lon → อ่านทั้งไฟล์
            else: kw["bbox"] = (SARA_BBOX[0] - .3, SARA_BBOX[1] - .3, SARA_BBOX[2] + .3, SARA_BBOX[3] + .3)
        except Exception:
            g = None
    if g is None: g = gpd.read_file(src, **kw)
    thai = lambda s: bool(re.search(r"[฀-๿]", str(s)))
    garbled = lambda s: bool(re.search(r"[À-ÿ]{2,}", str(s)))
    is_txt = lambda c: c != "geometry" and not pd.api.types.is_numeric_dtype(g[c])
    obj = [c for c in g.columns if is_txt(c)]
    if obj and not any(g[c].astype(str).map(thai).any() for c in obj) and any(g[c].astype(str).map(garbled).any() for c in obj):
        for enc in ("cp874", "tis-620"):
            try:
                g2 = gpd.read_file(src, encoding=enc, **kw)
                if any(g2[c].astype(str).map(thai).any() for c in obj if c in g2): g = g2; obj = [c for c in g.columns if is_txt(c)]; print(f"   อ่านภาษาไทยด้วย encoding={enc}"); break
            except Exception: pass
    g = g[g.geometry.notna() & g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    if g.empty: raise RuntimeError("ไม่มี polygon (อาจเป็นไฟล์จุด/เส้น)")
    if g.crs is None:
        x0, y0, x1, y1 = g.total_bounds
        if -180 <= x0 <= 180 and -90 <= y0 <= 90: g = g.set_crs("EPSG:4326")
        else: g = g.set_crs(UTM); print("   ⚠ ไฟล์ไม่มี .prj — สมมติเป็น UTM 47N (EPSG:32647)")
    g = g.to_crs("EPSG:4326")
    up = {c: c.upper() for c in g.columns}
    digits = lambda s: s.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    # ---- รหัสตำบล 6 หลัก ----
    code = None
    for c in g.columns:
        if c == "geometry": continue
        d = digits(g[c])
        if (d.str.len() == 6).mean() > 0.8 and d.str[:2].nunique() <= 80: code = d.where(d.str.len() == 6); break
    if code is None:
        pc = next((c for c in g.columns if re.fullmatch(r"(P|PV|PROV|PRO|CHANGWAT|ADM1)_?(CODE|ID|CD)", up[c])), None)
        ac = next((c for c in g.columns if re.fullmatch(r"(A|AP|AMP|AMPHOE|DIST|ADM2)_?(CODE|ID|CD)", up[c])), None)
        tc = next((c for c in g.columns if re.fullmatch(r"(T|TB|TAM|TAMBON|SUBDIST|ADM3)_?(CODE|ID|CD)", up[c])), None)
        if pc and ac and tc:
            p, a, t = digits(g[pc]).str[-2:].str.zfill(2), digits(g[ac]).str[-2:].str.zfill(2), digits(g[tc]).str[-2:].str.zfill(2)
            code = p + a + t
    # ---- ตัดเฉพาะจังหวัดสระบุรี ----
    n_all = len(g); keep = None
    if code is not None and (code.str[:2] == "19").sum() >= MIN_TAMBONS: keep = code.str[:2] == "19"
    if keep is None:
        for c in g.columns:
            if is_txt(c):
                k = g[c].astype(str).str.contains(r"สระบุรี|Saraburi", case=False, na=False)
                if MIN_TAMBONS <= k.sum() < len(g): keep = k; break
    if keep is None and n_all > 140:
        rp = g.representative_point(); w, s, e, n = SARA_BBOX
        keep = rp.x.between(w, e) & rp.y.between(s, n)
        print("   ⚠ ไม่พบคอลัมน์จังหวัด — ตัดด้วยกรอบพิกัดโดยประมาณ (ตรวจตำบลริมขอบจังหวัดอีกครั้ง)")
    if keep is not None:
        g = g[keep.values].copy(); code = code[keep.values] if code is not None else None
    # ---- ชื่อตำบล ----
    pick = lambda pats: next((c for p in pats for c in g.columns if re.fullmatch(p, up[c])), None)
    nc = pick([r"TAM_?NAMT", r"T_?NAME_?T", r"TB_?TN", r"TAMBON_?T(H)?", r"ADM3_?TH", r"NAME_?TH", r"TAM_?NAME", r"T_?NAME", r"TAMBON", r"ADM3_?EN", r"NAME_?3", r"NAME"])
    if nc is None:
        nc = next((c for c in g.columns if is_txt(c) and g[c].astype(str).map(thai).mean() > 0.8), None)
    ec = pick([r"TAM_?NAME_?E", r"TAM_?NAMEE", r"T_?NAME_?E", r"TB_?EN", r"ADM3_?EN", r"NAME_?EN", r"NAME_?3"])
    g["name"] = g[nc].astype(str).str.replace(r"^(ตำบล|ต\.)\s*", "", regex=True) if nc else [f"T{i+1:03d}" for i in range(len(g))]
    if ec and ec != nc: g["name_en"] = g[ec].astype(str)
    if code is not None: g["a_code"] = code.values
    g["tambon_id"] = g["a_code"] if "a_code" in g else g["name"]
    cols = ["tambon_id"] + (["a_code"] if "a_code" in g else []) + ["name"] + (["name_en"] if "name_en" in g else []) + ["geometry"]
    info = f"รหัส={'มี' if 'a_code' in g else 'ไม่มี'}, ชื่อ={nc}, ตัดจาก {n_all} → {len(g)} หน่วย"
    return g[cols].reset_index(drop=True), info

def load_tambons_drive():
    import time
    errs = []; t0 = time.time()
    for path in find_tambon_files()[:6]:
        short = path.replace("/content/drive/MyDrive/", "MyDrive/")
        try:
            g, info = read_tambon_file(path)
            if len(g) < MIN_TAMBONS: raise RuntimeError(f"ได้ {len(g)} หน่วย ({info})")
            print(f"   ไฟล์: {short} ({info}; {time.time() - t0:.0f} s)")
            if os.path.basename(path) != TAMBON_CACHE:
                try:                                      # ครั้งต่อไปอ่านสำเนานี้แทน (ไม่กี่วินาที)
                    cp = os.path.join(os.path.dirname(path), TAMBON_CACHE)
                    g.to_file(cp, driver="GPKG"); print(f"   บันทึกสำเนาเฉพาะสระบุรี → {cp.replace('/content/drive/MyDrive/', 'MyDrive/')}")
                except Exception as e:
                    print(f"   (บันทึกสำเนาไม่ได้: {str(e)[:60]})")
            return g, f"Drive: {os.path.basename(path)}"
        except Exception as e:
            errs.append(f"{os.path.basename(path)}: {str(e)[:70]}")
    raise RuntimeError("ไฟล์ใน Drive ใช้ไม่ได้ — " + " | ".join(errs))

MIN_TAMBONS = 60     # สระบุรีมี ~108–111 ตำบล ถ้าได้น้อยกว่านี้แสดงว่าได้ขอบเขตผิดระดับ (เช่น อำเภอ)

def clean_units(g):
    """ตัด polygon ซ้ำ และ polygon 'แม่' ที่ครอบหน่วยอื่น ≥ 2 หน่วย (เช่น อำเภอ/จังหวัดที่ปนมา)"""
    n0 = len(g)
    g = g[g.geometry.notna() & ~g.geometry.is_empty].copy()
    g["geometry"] = g.geometry.buffer(0)
    key = g.to_crs(UTM).geometry.apply(lambda x: (round(x.area / 1e4), round(x.centroid.x / 50), round(x.centroid.y / 50)))
    g = g[~key.duplicated()].reset_index(drop=True)
    rp = g.representative_point(); sidx = rp.sindex
    area = g.to_crs(UTM).area.values
    parent = []
    for i, geom in enumerate(g.geometry):
        inside = [j for j in sidx.query(geom, predicate="contains") if j != i and area[j] < 0.8 * area[i]]
        parent.append(len(inside) >= 2)               # ครอบหน่วยที่เล็กกว่าอย่างน้อย 2 หน่วย = ขอบเขตระดับบน
    g = g[~np.array(parent)].reset_index(drop=True)
    if "name" in g:
        g = g[~g["name"].astype(str).str.match(r"^(อำเภอ|จังหวัด|เขต)")].reset_index(drop=True)
    if len(g) != n0:
        print(f"   ทำความสะอาดขอบเขต: {n0} → {len(g)} หน่วย (ตัด polygon ซ้ำ/ระดับอำเภอ-จังหวัดที่ปนมา)")
    return g

KV_URL = "https://raw.githubusercontent.com/kongvut/thai-province-data/master/api/latest/sub_district.json"

def add_tambon_codes(g):
    """เติมรหัสตำบล 6 หลัก (มท.) ให้ขอบเขตที่ไม่มีรหัส (GADM/OSM): จับคู่ตำแหน่งตำบลก่อน แล้วจับคู่ชื่อ"""
    import re
    js = http_get(KV_URL, timeout=(15, 120)).json()
    kv = pd.DataFrame([{"a_code": str(t["id"]), "th": (t["name"] or {}).get("th", ""), "en": (t["name"] or {}).get("en", ""),
                        "lat": t.get("lat"), "lon": t.get("long")} for t in js if t.get("id")])
    pts = kv.dropna(subset=["lat", "lon"])
    pts = gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts.lon, pts.lat), crs="EPSG:4326")
    j = gpd.sjoin(pts, g[["geometry"]].reset_index().rename(columns={"index": "gi"}), predicate="within")
    prov = j.a_code.str[:2].mode().iloc[0]
    kv = kv[kv.a_code.str[:2] == prov]
    norm_th = lambda x: re.sub(r"^(ตำบล|ต\.)", "", str(x)).replace(" ", "").strip()
    norm_en = lambda x: re.sub(r"[^a-z]", "", str(x).lower())
    by_th = dict(zip(kv.th.map(norm_th), kv.a_code)); by_en = dict(zip(kv.en.map(norm_en), kv.a_code))
    code = pd.Series(index=g.index, dtype=object)
    for col, table, f in [("name", by_th, norm_th), ("name_en", by_en, norm_en), ("name", by_en, norm_en)]:
        if col in g:
            m = g[col].map(f).map(table); code = code.fillna(m)
    jj = j[j.a_code.str[:2] == prov].drop_duplicates("gi").set_index("gi")["a_code"]
    code = code.fillna(jj.reindex(g.index))
    g["a_code"] = code
    th = dict(zip(kv.a_code, kv.th)); g["name_th"] = g["a_code"].map(th)
    if g["name_th"].notna().mean() > 0.8: g["name"] = g["name_th"].fillna(g["name"])
    print(f"   เติมรหัสตำบล (มท.) ได้ {g.a_code.notna().sum()} / {len(g)} หน่วย (จังหวัดรหัส {prov})")
    return g.drop(columns="name_th")

def load_tambons():
    loaders = {"drive": load_tambons_drive, "cod": load_tambons_cod, "gadm": load_tambons_gadm, "osm": load_tambons_osm}
    first = BOUNDARY_SOURCE if BOUNDARY_SOURCE in loaders else "drive"
    order = [loaders[first]] + [f for k, f in loaders.items() if k != first]
    for f in order:
        try:
            g, src = f(); g = clean_units(g.reset_index(drop=True))
            if len(g) < MIN_TAMBONS:
                raise RuntimeError(f"ได้เพียง {len(g)} หน่วย — น่าจะไม่ใช่ระดับตำบล")
            if "a_code" not in g or g["a_code"].isna().mean() > 0.2:
                try:
                    g = add_tambon_codes(g)
                except Exception as e:
                    print(f"   ⚠ เติมรหัสตำบลไม่ได้: {str(e)[:80]} — การจับคู่ข้อมูล HDC จะใช้ชื่อแทน")
            print(f"✓ ขอบเขตตำบล: {len(g)} ตำบล  ({src})"); return g, src
        except Exception as e:
            print(f"   ✗ {f.__name__}: {str(e)[:120]}")
    raise RuntimeError("โหลดขอบเขตตำบลไม่สำเร็จ")

WORLDPOP_URLS = [
    "https://data.worldpop.org/GIS/Population/Global_2000_2020_1km_UNadj/2020/THA/tha_ppp_2020_1km_Aggregated_UNadj.tif",  # ~3 MB
    "https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/THA/tha_ppp_2020_UNadj.tif",                            # ~250 MB (100 m)
]

def add_population(g):
    """ดาวน์โหลด GeoTIFF ของ WorldPop ลงเครื่อง แล้วรวมประชากรรายตำบล (zonal sum)"""
    import rasterio
    from rasterio.mask import mask
    for u in [WORLDPOP_URLS[0]] + WORLDPOP_URLS:          # ไฟล์ 1 km ลอง 2 ครั้ง (server ตัดการเชื่อมต่อบ่อย) แล้วจึงใช้ 100 m
        fn = os.path.join("/content" if os.path.isdir("/content") else ".", u.split("/")[-1])
        try:
            if not (os.path.exists(fn) and os.path.getsize(fn) > 1e6):
                print(f"   ดาวน์โหลด {u.split('/')[-1]} ...")
                with requests.get(u, headers=UA, stream=True, timeout=(15, 600)) as r:
                    r.raise_for_status()
                    with open(fn, "wb") as f:
                        for chunk in r.iter_content(1 << 20): f.write(chunk)
            with rasterio.open(fn) as src:
                gg = g.to_crs(src.crs); pop = []
                for geom in gg.geometry:
                    arr, _ = mask(src, [geom], crop=True, all_touched=False, filled=True, nodata=0)
                    a = arr[0].astype(float); a[~np.isfinite(a) | (a < 0)] = 0; pop.append(a.sum())
            if sum(pop) <= 0:
                raise RuntimeError("ผลรวมประชากร = 0")
            g["pop"] = np.round(pop); print(f"✓ ประชากร WorldPop 2020: รวม {g['pop'].sum():,.0f} คน  ({u.split('/')[-1]})")
            return g, "WorldPop 2020 UN-adjusted (" + u.split("/")[-1] + ")"
        except Exception as e:
            print(f"   ✗ WorldPop {u.split('/')[-1]}: {type(e).__name__}: {str(e)[:100]}")
            if os.path.exists(fn) and os.path.getsize(fn) < 1e6: os.remove(fn)
    print("⚠ ไม่ได้ข้อมูลประชากร — ใช้พื้นที่แทนเป็นตัวหารสำรอง"); g["pop"] = np.nan
    return g, None

FIRMS_PRODUCTS = ["viirs-snpp", "viirs-noaa20", "modis"]       # ลำดับความชอบ (VIIRS 375 m ละเอียดกว่า MODIS 1 km)
FIRMS_API_SRC = {"viirs-snpp": ["VIIRS_SNPP_SP", "VIIRS_SNPP_NRT"], "viirs-noaa20": ["VIIRS_NOAA20_SP", "VIIRS_NOAA20_NRT"],
                 "modis": ["MODIS_SP", "MODIS_NRT"]}


# ---- WorldPop: อ่านเฉพาะกรอบพื้นที่จากไฟล์ 100 m ผ่าน HTTP (ไม่ต้องดาวน์โหลดทั้งประเทศ) → สำรอง: ไฟล์ 1 km ----
def _wp_cache_path(bounds, u):
    import hashlib
    d = globals().get("WP_CACHE_DIR") or ""
    if d.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
        d = os.path.join(OUTDIR, "WorldPop_cache")                     # Drive ไม่ได้ mount → เก็บใน OUTDIR
    d = d or os.path.join(OUTDIR, "WorldPop_cache"); os.makedirs(d, exist_ok=True)
    tag = hashlib.md5(str(np.round(bounds, 4)).encode()).hexdigest()[:8]
    return os.path.join(d, u.split("/")[-1].replace(".tif", f"_{tag}.tif"))

def worldpop_window(bounds, prefer_100m=True):
    """คืน (array ประชากรต่อเซลล์, affine transform, crs, คำอธิบายแหล่ง) เฉพาะกรอบ bounds = (W, S, E, N) องศา
    ลำดับ: ① ไฟล์ที่ตัดกรอบไว้แล้วใน Drive (WP_CACHE_DIR) → ② อ่านบางส่วนผ่าน HTTP → ③ ดาวน์โหลดทั้งไฟล์
    ได้ผลแล้วจะเก็บเฉพาะกรอบลง Drive (ไม่กี่ MB) — ครั้งต่อไปไม่ต้องดาวน์โหลด 263 MB อีก (ใช้ร่วมกันระหว่าง Notebook 06 และ 08)"""
    import rasterio
    from rasterio.windows import from_bounds
    urls = (WORLDPOP_URLS[1:] if prefer_100m else []) + WORLDPOP_URLS[:1]
    for u in urls:
        cp = _wp_cache_path(bounds, u)
        if os.path.exists(cp):
            with rasterio.open(cp) as src:
                arr = src.read(1).astype("float64"); tr = src.transform; crs = src.crs
            label = f"WorldPop 2020 UN-adjusted ({u.split('/')[-1]}, cache: {cp})"
            print(f"✓ {label}: {arr.shape[1]}×{arr.shape[0]} เซลล์, รวม {arr.sum():,.0f} คน")
            return arr, tr, crs, label
    for u in urls:
        for how in ("vsicurl", "download"):
            try:
                if how == "vsicurl":
                    path = "/vsicurl/" + u
                    env = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY="4", GDAL_HTTP_RETRY_DELAY="3",
                               CPL_VSIL_CURL_CHUNK_SIZE=str(1 << 20))
                else:
                    path = os.path.join("/content" if os.path.isdir("/content") else ".", u.split("/")[-1])
                    if not (os.path.exists(path) and os.path.getsize(path) > 1e6):
                        print(f"   ดาวน์โหลด {u.split('/')[-1]} ทั้งไฟล์ ...", flush=True)
                        with requests.get(u, headers=UA, stream=True, timeout=(15, 900)) as r:
                            r.raise_for_status()
                            with open(path + ".part", "wb") as f:
                                for ch in r.iter_content(1 << 22): f.write(ch)
                        os.replace(path + ".part", path)
                    env = {}
                with rasterio.Env(**env), rasterio.open(path) as src:
                    win = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
                    arr = src.read(1, window=win, boundless=True, fill_value=0).astype("float64")
                    tr = src.window_transform(win); crs = src.crs
                arr[~np.isfinite(arr) | (arr < 0)] = 0
                if arr.sum() <= 0: raise RuntimeError("ผลรวมประชากร = 0")
                label = f"WorldPop 2020 UN-adjusted ({u.split('/')[-1]}, {how})"
                print(f"✓ {label}: {arr.shape[1]}×{arr.shape[0]} เซลล์, รวม {arr.sum():,.0f} คน")
                try:                                                   # เก็บเฉพาะกรอบไว้ใช้ครั้งต่อไป
                    cp = _wp_cache_path(bounds, u)
                    with rasterio.open(cp, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1,
                                       dtype="float32", crs=crs, transform=tr, compress="deflate") as dst:
                        dst.write(arr.astype("float32"), 1)
                    print(f"   เก็บเฉพาะกรอบไว้ที่ {cp} ({os.path.getsize(cp) / 1e6:.1f} MB) — ครั้งต่อไปไม่ต้องดาวน์โหลด")
                    if how == "download" and os.path.getsize(path) > 1e8:
                        os.remove(path)                                # ลบไฟล์ทั้งประเทศ (263 MB) ออกจากเครื่อง Colab
                except Exception as e:
                    print(f"   ⚠ เก็บ cache ไม่ได้: {str(e)[:80]}")
                return arr, tr, crs, label
            except Exception as e:
                print(f"   ✗ WorldPop {u.split('/')[-1]} ({how}): {type(e).__name__}: {str(e)[:90]}")
    raise RuntimeError("อ่าน WorldPop ไม่ได้ — ลองรันเซลล์ใหม่ภายหลัง")

In [ ]:
# =====================================================================
#  ค่าตรวจวัด PM2.5 จริง (ฟังก์ชันชุดเดียวกับ Notebook 05 และ 09–12)
#  1) ไฟล์ PM2.5 รายวัน (เฉลี่ย 24 ชม.) ของ คพ. 2016–2026 ใน Google Drive (PCD_FOLDER_ID) — ทุกสถานี
#  2) Air4Thai API (กรมควบคุมมลพิษ) : รายชั่วโมงช่วงล่าสุด ~2–3 เดือน ที่ไฟล์ คพ. ยังไม่มี
#  ใช้เฉพาะค่าตรวจวัดจริง — ถ้าไม่มีข้อมูลจะหยุดพร้อมแจ้ง ไม่ใช้ค่าจากแบบจำลองแทน
# =====================================================================
A4T_META_URLS = ["http://air4thai.pcd.go.th/services/getNewAQI_JSON.php",
                 "http://air4thai.com/forweb/getAQI_JSON.php"]
# endpoint ข้อมูลย้อนหลังที่หน้าเว็บ "ข้อมูลย้อนหลัง" ของ Air4Thai ใช้ (ดู databuri, Code for Thailand)
A4T_HIST_BASES = ["http://air4thai.pcd.go.th/webV2/history/api/data.php",
                  "http://air4thai.pcd.go.th/webV3/history/api/data.php",
                  "http://air4thai.com/forweb/getHistoryData.php"]

def update_station_coords(stations):
    """อัปเดตพิกัดสถานีจากบริการ Air4Thai (ถ้าเข้าถึงได้)"""
    for u in A4T_META_URLS:
        try:
            js = http_get(u, timeout=(10, 40), tries=1).json()
            found = 0
            for s in js.get("stations", []):
                sid = str(s.get("stationID", "")).lower()
                if sid in stations:
                    stations[sid]["lat"] = float(s["lat"]); stations[sid]["lon"] = float(s["long"])
                    stations[sid]["name_th"] = s.get("nameTH", "") + " " + s.get("areaTH", "")
                    found += 1
            if found:
                print(f"✓ อัปเดตพิกัดจาก Air4Thai แล้ว {found} สถานี")
                return stations
        except Exception as e:
            print("  ⚠", u.split('/')[2], "→", str(e)[:90])
    print("⚠ เข้าถึงรายชื่อสถานี Air4Thai ไม่ได้ — ใช้พิกัดโดยประมาณใน CONFIG (ควรตรวจสอบก่อนรายงานผล)")
    return stations

def _find_records(js):
    if isinstance(js, list):
        if js and isinstance(js[0], dict) and "DATETIMEDATA" in js[0]:
            return js
        for x in js:
            r = _find_records(x)
            if r: return r
    elif isinstance(js, dict):
        for v in js.values():
            r = _find_records(v)
            if r: return r
    return None

# Air4Thai ตอบ {"result": ..., "error": ..., "stations": [...]} — ถ้า data ว่าง ให้ลองรูปแบบพารามิเตอร์อื่นอัตโนมัติ
A4T_VARIANTS = [
    {"param_map": {}, "etime": "24", "type": "hr", "upper": False},
    {"param_map": {}, "etime": "23", "type": "hr", "upper": False},
    {"param_map": {"PM25": "PM2.5"}, "etime": "24", "type": "hr", "upper": False},
    {"param_map": {}, "etime": "24", "type": "hr", "upper": True},
    {"param_map": {"PM25": "PM2.5"}, "etime": "23", "type": "hr", "upper": True},
]
# ผล smoke test: webV2 + พารามิเตอร์มาตรฐานใช้ได้ แต่ API เก็บข้อมูลย้อนหลังได้จำกัด (ข้อมูลเก่าตอบกลับว่าง)
A4T_OK = ("http://air4thai.pcd.go.th/webV2/history/api/data.php", A4T_VARIANTS[0])
A4T_EARLIEST = None      # วันแรกที่ API ยังมีข้อมูล (หาโดย air4thai_window)

def _a4t_has_data(sid, day):
    s = day.strftime("%Y-%m-%d"); e = (day + pd.Timedelta(days=2)).strftime("%Y-%m-%d")
    try:
        rec, _ = _a4t_get(_a4t_url(A4T_OK[0], sid, ("PM25",), s, e, A4T_OK[1]))
        return bool(rec) and any(r.get("PM25") not in (None, "", "-") for r in rec)
    except Exception:
        return False

def air4thai_window(sid="24t"):
    """หาว่า API ข้อมูลย้อนหลังของ Air4Thai มีข้อมูลย้อนไปได้ถึงเมื่อไร (ตรวจทีละช่วงเดือน)"""
    global A4T_EARLIEST
    if A4T_EARLIEST is not None: return A4T_EARLIEST
    today = pd.Timestamp.now().normalize(); last_ok = None
    for k in [1, 2, 3, 6, 9, 12, 18, 24, 36, 60, 120]:
        d = today - pd.DateOffset(months=k)
        if _a4t_has_data(sid, d): last_ok = d
        else: break
    if last_ok is not None:                  # ค้นละเอียดระดับวันระหว่างเดือนที่มีข้อมูลกับเดือนถัดไปที่ไม่มี
        lo, hi = last_ok, last_ok - pd.DateOffset(months=3)
        if not _a4t_has_data(sid, hi):
            for _ in range(6):
                mid = lo - (lo - hi) / 2
                if _a4t_has_data(sid, mid.normalize()): lo = mid.normalize()
                else: hi = mid.normalize()
        last_ok = lo
    A4T_EARLIEST = last_ok if last_ok is not None else today - pd.Timedelta(days=7)
    print(f"Air4Thai: API ข้อมูลย้อนหลังมีถึงประมาณ {A4T_EARLIEST:%Y-%m-%d} (~{(today - A4T_EARLIEST).days} วัน) — "
          "ช่วงที่เก่ากว่านี้มาจากไฟล์ คพ.")
    return A4T_EARLIEST

def resolve_period(start, end, sid="24t", max_days=None):
    """start/end = None → อัตโนมัติ: start = วันแรกที่สถานีมีข้อมูลในไฟล์ คพ., end = เมื่อวาน (Air4Thai API มีข้อมูลล่าสุด)
    ถ้าไม่มีไฟล์ คพ. จะใช้ได้เฉพาะช่วงที่ API ยังเก็บไว้ (~2 เดือน)"""
    T = load_pcd_table() if _pcd_path() else None
    ser = T[sid.lower()].dropna() if T is not None and sid.lower() in T else None
    e = pd.Timestamp(end) if end else pd.Timestamp.now().normalize() - pd.Timedelta(days=1)
    if start:
        s = pd.Timestamp(start)
    elif ser is not None:
        s = ser.index.min()
    else:
        s = air4thai_window(sid)
    if max_days:
        s = max(s, e - pd.Timedelta(days=max_days - 1))
    if ser is None:
        aw = air4thai_window(sid)
        if s < aw:
            print(f"⚠ ไม่มีไฟล์ คพ. — ข้อมูลจริงก่อน {aw:%Y-%m-%d} จะไม่มี (ตั้ง PCD_FOLDER_ID หรือ PCD_PATH)")
    else:
        aw = air4thai_window(sid)
        gap = (aw - ser.index.max()).days - 1
        print(f"ข้อมูลจริง {sid.upper()}: ไฟล์ คพ. {ser.index.min():%Y-%m-%d} – {ser.index.max():%Y-%m-%d}"
              f" | Air4Thai API {aw:%Y-%m-%d} – ปัจจุบัน" + (f" | ช่องว่างที่ไม่มีข้อมูล {gap} วัน" if gap > 0 else ""))
    print(f"ช่วงเวลาที่ใช้: {s:%Y-%m-%d} ถึง {e:%Y-%m-%d}")
    return s.strftime("%Y-%m-%d"), e.strftime("%Y-%m-%d")

def _a4t_url(base, sid, params, s, e, v):
    ps = ",".join(v["param_map"].get(p, p) for p in params)
    return (f"{base}?stationID={sid.upper() if v['upper'] else sid}&param={ps}&type={v['type']}"
            f"&sdate={s}&edate={e}&stime=00&etime={v['etime']}")

def _a4t_get(url):
    js = http_get(url, timeout=(10, 120), tries=2, quiet=True).json()
    return _find_records(js), (js.get("result"), js.get("error")) if isinstance(js, dict) else (None, None)

def _a4t_discover(sid, params, s, e):
    global A4T_OK
    for base in A4T_HIST_BASES:
        for v in A4T_VARIANTS:
            try:
                rec, info = _a4t_get(_a4t_url(base, sid, params, s, e, v))
            except Exception as ex:
                print(f"     {'/'.join(base.split('/')[3:])}: {type(ex).__name__}: {str(ex)[:80]}"); break
            if rec:
                A4T_OK = (base, v); print(f"     ✓ ใช้ {'/'.join(base.split('/')[3:])} พารามิเตอร์ {v}"); return rec
            print(f"     {'/'.join(base.split('/')[3:])} {v}: ไม่มีข้อมูล (result={info[0]!r}, error={str(info[1])[:80]!r})")
    return None

def fetch_air4thai_hourly(sid, start, end, params=("PM25", "WS", "WD")):
    """PM2.5 (+ WS/WD) รายชั่วโมงจาก Air4Thai — ดึงทีละเดือน"""
    frames, n_err, n_empty = [], 0, 0
    earliest = air4thai_window()
    for s, e in month_chunks(start, end):
        if pd.Timestamp(e) < earliest:
            n_empty += 1; continue                      # เก่ากว่าที่ API เก็บไว้ → ข้าม (ไม่เสียเวลา)
        rec = None
        try:
            rec, info = _a4t_get(_a4t_url(A4T_OK[0], sid, params, s, e, A4T_OK[1]))
            if not rec and not frames and n_err == 0 and n_empty == 0:
                rec = _a4t_discover(sid, params, s, e)
        except Exception as ex:
            n_err += 1; print(f"     {type(ex).__name__}: {str(ex)[:80]}")
        if rec: frames.append(pd.DataFrame(rec))
        else: n_empty += 1
        print(f"   Air4Thai {sid} {s}..{e}: {'✓' if rec else '✗ (ว่าง)'}")
        if not frames and n_err >= 2:
            print("   ✗ เชื่อมต่อ Air4Thai API ไม่ได้ → ใช้เฉพาะไฟล์ คพ."); return None
    if n_empty:
        print(f"   {sid}: Air4Thai API มีข้อมูลตั้งแต่ ~{earliest:%Y-%m-%d} — {n_empty} เดือนก่อนหน้านั้นใช้ไฟล์ คพ. (ถ้ามี)")
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    df["time"] = pd.to_datetime(df["DATETIMEDATA"], errors="coerce")
    df = df.dropna(subset=["time"]).drop_duplicates("time").set_index("time").sort_index()
    df = df.rename(columns={"PM2.5": "PM25"})
    out = pd.DataFrame(index=df.index)
    for p in params:
        if p in df:
            out[p] = pd.to_numeric(df[p], errors="coerce")
    return out.where(out >= 0)          # ค่าติดลบ (-1, -99) = missing

# ---- ข้อมูล PM2.5 ย้อนหลังจากไฟล์ของกรมควบคุมมลพิษ (ใช้เมื่อช่วงเวลาเก่ากว่าที่ API ให้ ~2 เดือน) ----
# PCD_PATH = โฟลเดอร์ใน Drive ที่มีไฟล์ PM25_2016.xlsx … PM25_2026.xlsx (ค่าเฉลี่ย 24 ชม. ทุกสถานี, ชีตแรก: Date + รหัสสถานี)
#            หรือไฟล์เดียว (.xlsx/.csv) ก็ได้
_PCD_CACHE = {}

PCD_LOCAL_DIR = "/content/pcd_pm25" if os.path.isdir("/content") else "./pcd_pm25"
PCD_MOUNT_GUESS = "/content/drive/MyDrive/3-ColabProjects/Training/PCD_PM25"
_PCD_DIR = None

def download_pcd_folder(folder_id, dest=PCD_LOCAL_DIR):
    """ดาวน์โหลดไฟล์ทุกไฟล์ในโฟลเดอร์ Google Drive (ตาม folder ID) มาไว้ในเครื่อง Colab
    1) Google Drive API ด้วยบัญชีของคุณ (Colab จะถามให้ลงชื่อเข้าใช้ครั้งแรก) — ใช้ได้แม้โฟลเดอร์เป็นส่วนตัว
    2) gdown (ถ้าโฟลเดอร์แชร์แบบ "ทุกคนที่มีลิงก์")
    3) mount Drive แล้วอ่านจาก path ที่รู้จัก"""
    import glob, shutil
    os.makedirs(dest, exist_ok=True)
    have = glob.glob(os.path.join(dest, "*.xls*")) + glob.glob(os.path.join(dest, "*.csv"))
    if have:
        print(f"✓ ใช้ไฟล์ คพ. ที่ดาวน์โหลดไว้แล้ว {len(have)} ไฟล์ ({dest})"); return dest
    try:
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        svc = build("drive", "v3", cache_discovery=False)
        files, tok = [], None
        while True:
            r = svc.files().list(q=f"'{folder_id}' in parents and trashed=false", pageSize=200, pageToken=tok,
                                 fields="nextPageToken, files(id, name, mimeType, size)",
                                 supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            files += r.get("files", []); tok = r.get("nextPageToken")
            if not tok: break
        n = 0
        for f in files:
            if not f["name"].lower().endswith((".xlsx", ".xls", ".csv")): continue
            with io.FileIO(os.path.join(dest, f["name"]), "wb") as fh:
                dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=f["id"], supportsAllDrives=True)); done = False
                while not done: _, done = dl.next_chunk()
            n += 1
        if n:
            print(f"✓ ดาวน์โหลดไฟล์ คพ. จาก Google Drive (Drive API) {n} ไฟล์ → {dest}"); return dest
        print("   ⚠ Drive API: ไม่พบไฟล์ .xlsx/.csv ในโฟลเดอร์")
    except Exception as e:
        print(f"   ✗ Drive API: {type(e).__name__}: {str(e)[:120]}")
    try:
        import importlib.util
        if importlib.util.find_spec("gdown") is None:
            subprocess.run(f"{sys.executable} -m pip -q install gdown", shell=True)
        import gdown
        got = gdown.download_folder(id=folder_id, output=dest, quiet=True) or []
        if got:
            print(f"✓ ดาวน์โหลดไฟล์ คพ. ด้วย gdown {len(got)} ไฟล์"); return dest
    except Exception as e:
        print(f"   ✗ gdown: {type(e).__name__}: {str(e)[:120]}")
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
        if os.path.isdir(PCD_MOUNT_GUESS):
            print(f"✓ ใช้ไฟล์ คพ. จาก Drive ที่ mount: {PCD_MOUNT_GUESS}"); return PCD_MOUNT_GUESS
    except Exception as e:
        print(f"   ✗ mount Drive: {type(e).__name__}: {str(e)[:120]}")
    raise RuntimeError("โหลดไฟล์ PM2.5 ของ คพ. จาก Google Drive ไม่ได้ — ตรวจ PCD_FOLDER_ID หรือใส่ PCD_PATH เอง")

def _pcd_path():
    global _PCD_DIR
    p = globals().get("PCD_PATH") or globals().get("PCD_STATION_CSV") or ""
    if p: return p
    fid = globals().get("PCD_FOLDER_ID", "")
    if not fid: return ""
    if _PCD_DIR is None:
        _PCD_DIR = download_pcd_folder(fid)
    return _PCD_DIR

def load_pcd_table(path=None):
    """อ่านไฟล์ PCD ทั้งหมด → DataFrame รายวัน (index = วันที่, คอลัมน์ = รหัสสถานีตัวพิมพ์เล็ก เช่น '24t')"""
    import glob
    path = path or _pcd_path()
    if not path: return None
    if path in _PCD_CACHE: return _PCD_CACHE[path]
    if path.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive; drive.mount("/content/drive")
    files = sorted(glob.glob(os.path.join(path, "*.xlsx")) + glob.glob(os.path.join(path, "*.csv"))) if os.path.isdir(path) \
            else sorted(glob.glob(path))
    frames = []
    for f in files:
        try:
            d = pd.read_csv(f, encoding="utf-8-sig") if f.lower().endswith(".csv") else pd.read_excel(f, sheet_name=0)
        except Exception as e:
            print(f"   ⚠ อ่าน {os.path.basename(f)} ไม่ได้: {e}"); continue
        tcol = d.columns[0]
        d["_date"] = pd.to_datetime(d[tcol], errors="coerce")
        d = d[d["_date"].notna()].set_index("_date").drop(columns=[tcol])
        d.columns = [str(c).strip().lower() for c in d.columns]
        d = d.apply(pd.to_numeric, errors="coerce")             # 'N/A', หมายเหตุท้ายตาราง → NaN
        frames.append(d)
    if not frames:
        print(f"   ⚠ ไม่พบไฟล์ PCD ที่ {path}"); return None
    T = pd.concat(frames).sort_index(); T = T[~T.index.duplicated(keep="last")].where(lambda x: x >= 0)
    T.index = T.index.normalize(); T.index.name = "date"
    _PCD_CACHE[path] = T
    print(f"✓ ไฟล์ PCD: {len(files)} ไฟล์, {T.shape[1]} สถานี, {T.index.min():%Y-%m-%d} ถึง {T.index.max():%Y-%m-%d}")
    return T

def load_pcd_csv(path, sid):
    T = load_pcd_table(path)
    if T is None or sid.lower() not in T: return None
    ser = T[sid.lower()].dropna().rename("PM25")
    print(f"   ✓ PCD {sid}: {len(ser):,} วัน ({ser.index.min():%Y-%m-%d}–{ser.index.max():%Y-%m-%d})")
    return ser, True

def fetch_pm25_measured(sid, start, end, hourly_index=None, a4t="fetch"):
    """PM2.5 ค่าตรวจวัดจริง → (Series รายชั่วโมง, คำอธิบายแหล่ง)
    มีไฟล์ คพ.: ใช้ค่าเฉลี่ย 24 ชม. ของ คพ. + ค่าเฉลี่ยรายวันจาก Air4Thai API เฉพาะวันที่ไฟล์ยังไม่มี
                แล้วกระจายค่ารายวันให้ทุกชั่วโมงของวันนั้น (ความละเอียดเดียวกันตลอดช่วง)
    ไม่มีไฟล์ คพ.: ใช้ค่ารายชั่วโมงจาก Air4Thai API (~2 เดือนล่าสุด)"""
    if isinstance(a4t, str): a4t = fetch_air4thai_hourly(sid, start, end, params=("PM25",))
    api = a4t["PM25"] if (a4t is not None and "PM25" in a4t and a4t["PM25"].notna().any()) else None
    pcd = load_pcd_csv(_pcd_path(), sid) if _pcd_path() else None
    if pcd is None:
        if api is None: return None, None
        return api, f"Air4Thai API hourly ({api.notna().sum():,} h) (measured)"
    ser = pcd[0]; label = [f"PCD daily file ({ser.notna().sum():,} d)"]
    if api is not None:
        cnt = api.resample("D").count(); apid = api.resample("D").mean().where(cnt >= 18).dropna()
        new = apid[~apid.index.isin(ser.index)]
        if len(new):
            ser = pd.concat([ser, new]).sort_index(); label.append(f"Air4Thai API daily mean ({len(new)} d)")
    ser = ser[(ser.index >= pd.Timestamp(start)) & (ser.index <= pd.Timestamp(end))]
    idx = hourly_index if hourly_index is not None else pd.date_range(start, pd.Timestamp(end) + pd.Timedelta(hours=23), freq="h")
    out = ser.reindex(pd.DatetimeIndex(idx).normalize()).set_axis(idx)
    return out, " + ".join(label) + " → 24-h mean applied to each hour (measured)"


KNOWN_STATIONS = {"24t": ("สถานีตำรวจภูธรตำบลหน้าพระลาน", 14.685833, 100.871996),
                  "25t": ("สถานีดับเพลิงพระลักษณ์", 14.526343, 100.926047)}

def haversine_km(lat0, lon0, lat1, lon1):
    p0, p1 = np.deg2rad(lat0), np.deg2rad(lat1)
    a = np.sin((p1 - p0) / 2) ** 2 + np.cos(p0) * np.cos(p1) * np.sin(np.deg2rad(np.asarray(lon1) - lon0) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

def station_meta():
    """รายชื่อ + พิกัดสถานีทั้งประเทศจาก Air4Thai (ถ้าเข้าไม่ได้ ใช้พิกัด 24T/25T ที่รู้อยู่แล้ว)"""
    for u in A4T_META_URLS:
        try:
            js = http_get(u, timeout=(10, 60), tries=2).json()
            st = pd.DataFrame([{"sid": str(s["stationID"]).lower(), "name": s.get("nameTH", ""), "area": s.get("areaTH", ""),
                                "lat": float(s["lat"]), "lon": float(s["long"])} for s in js.get("stations", [])])
            if len(st):
                print(f"✓ พิกัดสถานีจาก Air4Thai: {len(st)} สถานี")
                return st
        except Exception as e:
            print("   ✗", u.split("/")[2], str(e)[:80])
    print("⚠ เข้าถึงรายชื่อสถานี Air4Thai ไม่ได้ — ใช้เฉพาะพิกัด 24T/25T ที่บันทึกไว้")
    return pd.DataFrame([{"sid": k, "name": v[0], "area": "สระบุรี", "lat": v[1], "lon": v[2]} for k, v in KNOWN_STATIONS.items()])


SEASON_ORDER = ["Winter", "Summer", "Rainy"]
SEASON_TH = {"Winter": "หนาว", "Summer": "ร้อน", "Rainy": "ฝน"}

def tmd_season(dates):
    """ฤดูตามกรมอุตุนิยมวิทยา: ร้อน 16 ก.พ.–15 พ.ค., ฝน 16 พ.ค.–15 ต.ค., หนาว 16 ต.ค.–15 ก.พ."""
    dates = pd.to_datetime(pd.Series(dates))
    md_ = dates.dt.month * 100 + dates.dt.day
    return pd.Series(np.select([md_.between(216, 515), md_.between(516, 1015)], ["Summer", "Rainy"], "Winter"),
                     index=dates.index)

def season_year(dates):
    """ปีของฤดู: ฤดูหนาวคร่อมปี (16 ต.ค.–15 ก.พ.) นับเป็นปีที่ฤดูสิ้นสุด เช่น ต.ค. 2023–ก.พ. 2024 = หนาว 2024"""
    dates = pd.to_datetime(pd.Series(dates))
    md_ = dates.dt.month * 100 + dates.dt.day
    return dates.dt.year + (md_ >= 1016).astype(int)


def save_table(df, name, index=False):
    """บันทึกตารางลง OUTDIR (ถ้า SAVE_TO_DRIVE=True คือ Drive: MyDrive/PM25_Saraburi_notebooks)"""
    path = os.path.join(OUTDIR, name)
    if name.endswith(".gpkg"):
        df.to_file(path, driver="GPKG")
    else:
        df.to_csv(path, index=index, encoding="utf-8-sig")
    return path

In [ ]:
# =====================================================================
#  ข้อมูล PM2.5 จากดาวเทียม: SatPM2.5 V6.GL.03 (ACAG, Washington University in St. Louis)
#  เก็บใน AWS Open Data (s3://satpmdata, เข้าถึงแบบสาธารณะ) — อ่านเฉพาะกรอบพื้นที่ แล้วเก็บ cache ลง Drive
# =====================================================================
import re
import xml.etree.ElementTree as ET
import xarray as xr

S3_NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

def s3_list(prefix, delimiter=None):
    """รายการไฟล์ใน bucket สาธารณะด้วย ListObjectsV2 (ไม่ต้องมีบัญชี AWS)"""
    keys, prefixes, token = [], [], None
    while True:
        params = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
        if delimiter: params["delimiter"] = delimiter
        if token: params["continuation-token"] = token
        root = ET.fromstring(http_get(SAT_S3 + "/", params=params, timeout=(15, 60)).content)
        keys += [(c.find("s3:Key", S3_NS).text, int(c.find("s3:Size", S3_NS).text)) for c in root.findall("s3:Contents", S3_NS)]
        prefixes += [p.find("s3:Prefix", S3_NS).text for p in root.findall("s3:CommonPrefixes", S3_NS)]
        if (root.findtext("s3:IsTruncated", default="false", namespaces=S3_NS) or "false").lower() != "true": break
        token = root.findtext("s3:NextContinuationToken", namespaces=S3_NS)
    return keys, prefixes

def _period_key(name):
    """V6GL03.CNNPM25.AS.201501-201501.nc → '201501' (รายเดือน) ; ...201501-201512.nc → '2015' (รายปี)"""
    m = re.search(r"(\d{6})-(\d{6})\.nc$", name)
    if not m: return None
    a, b = m.groups()
    if a == b: return a
    if a.endswith("01") and b.endswith("12") and a[:4] == b[:4]: return a[:4]
    return None


def sat_list_months():
    """{'YYYYMM': url หรือ path} ของไฟล์รายเดือน (หรือ {'YYYY': ...} เมื่อ SAT_TEMPORAL='Annual')"""
    want_monthly = SAT_TEMPORAL.lower().startswith("m")
    out = {}
    if SAT_LOCAL_DIR:
        import glob
        if SAT_LOCAL_DIR.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
            from google.colab import drive; drive.mount("/content/drive")
        for f in sorted(glob.glob(os.path.join(SAT_LOCAL_DIR, "**", "*.nc"), recursive=True)):
            k = _period_key(os.path.basename(f))
            if k and (len(k) == 6) == want_monthly: out[k] = f
        print(f"✓ ไฟล์ดาวเทียมใน SAT_LOCAL_DIR: {len(out)} ไฟล์")
        return out
    # โครงสร้าง bucket: V6GL03/CNNPM25/... (0.01°) และ V6GL03/CNNPM25c0p10/... (0.1°) — ชั้นย่อยอาจเป็นภูมิภาค/รายเดือน/ปี
    # จึง list ทุกไฟล์ใต้โฟลเดอร์ผลิตภัณฑ์ แล้วเลือกจากชื่อไฟล์ (ภูมิภาค + ช่วงเวลา) แทนการเดา path
    prefix = f"{SAT_VERSION}/{SAT_PRODUCT}/"
    try:
        keys, _ = s3_list(prefix)
    except Exception as e:
        keys = []; print(f"   ✗ list {prefix}: {type(e).__name__}: {str(e)[:90]}")
    keys = [(k, sz) for k, sz in keys if k.endswith(".nc") and f"/{SAT_PRODUCT}/" in f"/{k}"]
    print(f"   {prefix}: พบไฟล์ .nc {len(keys):,} ไฟล์")
    for region in SAT_REGIONS:
        tok = re.compile(rf"(^|[./_-]){region}([./_-])")
        cand = {}
        for k, size in keys:
            fname = k.split("/")[-1]; pk = _period_key(fname)
            if pk and (len(pk) == 6) == want_monthly and (tok.search(fname) or f"/{region}/" in f"/{k}"):
                cand.setdefault(pk, (f"{SAT_S3}/{k}", size))
        if cand:
            sizes = [v[1] for v in cand.values()]
            ex = next(iter(cand.values()))[0].replace(SAT_S3 + "/", "")
            print(f"✓ {SAT_PRODUCT} ภูมิภาค {region}: {len(cand)} ไฟล์ ({min(cand)}–{max(cand)}), ~{np.median(sizes) / 1e6:.0f} MB/ไฟล์ | ตัวอย่าง {ex}")
            return {k: v[0] for k, v in cand.items()}
        print(f"   ไม่พบไฟล์{SAT_TEMPORAL}ของภูมิภาค {region}")
    if keys:                                 # ชื่อไฟล์ไม่มีรหัสภูมิภาค → ใช้ได้ถ้าแต่ละช่วงเวลามีไฟล์เดียว
        cand = {}
        for k, size in keys:
            pk = _period_key(k.split("/")[-1])
            if pk and (len(pk) == 6) == want_monthly: cand.setdefault(pk, []).append((f"{SAT_S3}/{k}", size))
        if cand and all(len(v) == 1 for v in cand.values()):
            print(f"✓ {SAT_PRODUCT}: {len(cand)} ไฟล์ ({min(cand)}–{max(cand)}) (ชื่อไฟล์ไม่ระบุภูมิภาค)")
            return {k: v[0][0] for k, v in cand.items()}
        print("ตัวอย่างไฟล์ที่พบ:", [k for k, _ in keys[:8]])
    # ช่วยหาโฟลเดอร์ที่ถูกต้องเมื่อโครงสร้าง bucket เปลี่ยน
    try:
        _, top = s3_list("", delimiter="/"); print("โฟลเดอร์บนสุดของ bucket:", top)
        _, sub = s3_list(f"{SAT_VERSION}/", delimiter="/"); print(f"ใน {SAT_VERSION}/:", sub)
        _, sub2 = s3_list(prefix, delimiter="/"); print(f"ใน {prefix}:", sub2[:20])
    except Exception as e:
        print("   list bucket ไม่ได้:", str(e)[:90])
    raise RuntimeError("ไม่พบไฟล์ดาวเทียม — ตรวจ SAT_VERSION / SAT_PRODUCT / SAT_REGIONS ตามรายการไฟล์และโฟลเดอร์ข้างบน "
                       "หรือดาวน์โหลดไฟล์ .nc จาก https://sites.wustl.edu/acag/datasets/surface-pm2-5/ ไว้ใน Drive แล้วใส่ SAT_LOCAL_DIR")

def _standardise(ds, bounds):
    ren = {c: n for c, n in [("latitude", "lat"), ("longitude", "lon"), ("Latitude", "lat"), ("Longitude", "lon")] if c in ds.dims or c in ds.coords}
    ds = ds.rename(ren)
    var = next(v for v in ds.data_vars if {"lat", "lon"} <= set(ds[v].dims))
    da = ds[var]
    if da.ndim > 2: da = da.isel({d: 0 for d in da.dims if d not in ("lat", "lon")})
    w, s, e, n = bounds
    lat_sl = slice(s, n) if float(da.lat[0]) < float(da.lat[-1]) else slice(n, s)
    da = da.sel(lat=lat_sl, lon=slice(w, e)).load().sortby("lat")
    return da.astype("float32").rename("pm25")

def sat_open_month(key, src, bounds):
    """อ่านไฟล์ 1 เดือนเฉพาะกรอบ bounds — ① cache ใน Drive → ② อ่านบางส่วนผ่าน HTTP (h5netcdf) → ③ ดาวน์โหลดทั้งไฟล์"""
    import hashlib
    tag = hashlib.md5(str(np.round(bounds, 3)).encode()).hexdigest()[:8]
    cf = os.path.join(SAT_CACHE, f"{SAT_VERSION}_{key}_{tag}.nc")
    if os.path.exists(cf):
        return xr.open_dataarray(cf).load()
    da = None
    if str(src).startswith("http"):
        try:
            import fsspec
            with fsspec.open(src, "rb", block_size=4 << 20) as f:
                with xr.open_dataset(f, engine="h5netcdf") as ds:
                    da = _standardise(ds, bounds)
        except Exception as e:
            if not globals().get("_SAT_WARNED"):
                print(f"   (อ่านบางส่วนผ่าน HTTP ไม่ได้: {type(e).__name__} — ดาวน์โหลดทั้งไฟล์แทน)"); globals()["_SAT_WARNED"] = True
        if da is None:
            tmp = os.path.join("/content" if os.path.isdir("/content") else ".", "sat_tmp.nc")
            with requests.get(src, headers=UA, stream=True, timeout=(15, 900)) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for ch in r.iter_content(1 << 22): f.write(ch)
            with xr.open_dataset(tmp) as ds:
                da = _standardise(ds, bounds)
            os.remove(tmp)
    else:
        with xr.open_dataset(src) as ds:
            da = _standardise(ds, bounds)
    da.to_netcdf(cf)
    return da

## 1. ขอบเขตตำบลและประชากร

- **ขอบเขตตำบล:** OCHA COD-AB Thailand (มีรหัสตำบล 6 หลักตรงกับ HDC) ถ้าเข้าไม่ได้จะใช้ GADM 4.1 แล้วเติมรหัสตำบลอัตโนมัติ (ชุดเดียวกับ Notebook 13–20)
- **ประชากร:** WorldPop 2020 (UN-adjusted) ความละเอียด 100 m อ่านเฉพาะกรอบจังหวัดผ่าน HTTP — ใช้ถ่วงน้ำหนักค่า PM2.5 ตามที่คนอยู่จริง

In [ ]:
TAMBON, BND_SRC = load_tambons()
TAMBON = TAMBON.reset_index(drop=True)
TAMBON["tidx"] = np.arange(len(TAMBON))
TAMBON["area_km2"] = TAMBON.to_crs(UTM).area / 1e6
w, s, e, n = TAMBON.total_bounds
PROV_BOUNDS = (w - 0.02, s - 0.02, e + 0.02, n + 0.02)
WP, WP_TR, WP_CRS, POP_SRC = worldpop_window(PROV_BOUNDS, prefer_100m=WORLDPOP_100M)

from rasterio import features
TID = features.rasterize(((g, i + 1) for i, g in zip(TAMBON.tidx, TAMBON.to_crs(WP_CRS).geometry)),
                         out_shape=WP.shape, transform=WP_TR, fill=0, dtype="int32") - 1      # −1 = นอกจังหวัด
TAMBON["pop"] = np.bincount(TID[TID >= 0], weights=WP[TID >= 0], minlength=len(TAMBON))
print(f"✓ {len(TAMBON)} ตำบล ({BND_SRC}) ประชากรรวม {TAMBON['pop'].sum():,.0f} คน")
TAMBON.drop(columns="geometry").head()

## 2. PM2.5 จากดาวเทียมรายเดือน (เฉพาะกรอบพื้นที่)

กรอบที่อ่าน = จังหวัดสระบุรี + สถานี คพ. ในรัศมี 200 km (สำหรับตรวจสอบ) — ประมาณ 400 × 400 เซลล์ต่อเดือน
ไฟล์ต้นฉบับเป็นทั้งทวีป จึง **อ่านเฉพาะส่วนที่ต้องการผ่าน HTTP** ถ้าเซิร์ฟเวอร์ไม่รองรับจะดาวน์โหลดทั้งไฟล์แล้วตัด (ช้ากว่า)
ทุกเดือนที่อ่านแล้วถูกเก็บใน `SAT_CACHE` ใน Drive รันครั้งต่อไปไม่ต้องดึงใหม่

In [ ]:
META = station_meta()
T = load_pcd_table()
META = META[META.sid.isin(T.columns)].copy() if T is not None else META
META["dist_km"] = haversine_km(CENTER[0], CENTER[1], META.lat.values, META.lon.values)
META = META[(META.dist_km <= VALID_RADIUS_KM) & ~META.sid.str.startswith("m")].reset_index(drop=True)
bw = min(PROV_BOUNDS[0], META.lon.min() - .05); bs = min(PROV_BOUNDS[1], META.lat.min() - .05)
be = max(PROV_BOUNDS[2], META.lon.max() + .05); bn = max(PROV_BOUNDS[3], META.lat.max() + .05)
SAT_BOUNDS = (round(bw, 2), round(bs, 2), round(be, 2), round(bn, 2))
print(f"สถานีสำหรับตรวจสอบ: {len(META)} สถานี | กรอบดาวเทียม {SAT_BOUNDS}")

FILES = sat_list_months()
want = [k for k in sorted(FILES) if (SAT_START.replace("-", "")[:len(k)] <= k <= SAT_END.replace("-", "")[:len(k)])]
print(f"ช่วงที่ใช้: {want[0]}–{want[-1]} ({len(want)} ไฟล์)")
SAT = {}
t0 = time.time()
for i, k in enumerate(want):
    try:
        SAT[k] = sat_open_month(k, FILES[k], SAT_BOUNDS).sortby("lat").sortby("lon")   # lat/lon จากน้อยไปมาก
    except Exception as e:
        print(f"   ✗ {k}: {type(e).__name__}: {str(e)[:100]}")
    if i == 0 or (i + 1) % 12 == 0:
        print(f"   {k} ✓  ({i + 1}/{len(want)}, {time.time() - t0:.0f} s)", flush=True)
if not SAT:
    raise RuntimeError("ไม่ได้ข้อมูลดาวเทียมเลย — ดูข้อความ error ข้างบน")
keys = sorted(SAT)
LAT = SAT[keys[0]].lat.values; LON = SAT[keys[0]].lon.values
RES = float(np.median(np.diff(LON)))
CUBE = np.stack([SAT[k].reindex(lat=LAT, lon=LON, method="nearest", tolerance=RES / 4).values for k in keys])   # [เวลา, lat, lon]
TIME = pd.to_datetime([f"{k[:4]}-{k[4:6] if len(k) == 6 else '07'}-01" for k in keys])
print(f"✓ ข้อมูลดาวเทียม {len(keys)} ช่วงเวลา, กริด {len(LAT)}×{len(LON)} ที่ {RES:.3f}° | ค่าเฉลี่ย {np.nanmean(CUBE):.1f} µg/m³")

## 3. Zonal statistics: จากกริดเป็นค่ารายตำบล

**วิธีที่ 1 — ถ่วงพื้นที่ซ้อนทับ** (ทุกตำบลได้ค่า แม้ตำบลเล็กกว่าเซลล์)

$$\overline{PM}_i^{\,area}=\frac{\sum_c A_{ic}\,PM_c}{\sum_c A_{ic}}$$

**วิธีที่ 2 — ถ่วงประชากร** (ตัวแปรการสัมผัสหลักในโครงร่าง: ค่าที่ "คน" ในตำบลได้รับ ไม่ใช่ค่าของพื้นที่ป่า/เขา)

$$\overline{PM}_i^{\,pop}=\frac{\sum_c P_{ic}\,PM_c}{\sum_c P_{ic}}$$

$A_{ic}$ = พื้นที่ที่เซลล์ดาวเทียม $c$ ซ้อนทับตำบล $i$ · $P_{ic}$ = ประชากร WorldPop 100 m ที่อยู่ในตำบล $i$ และเซลล์ $c$

รูปร่างตำบลและกริดไม่เปลี่ยนตามเวลา จึงสร้าง **เมทริกซ์น้ำหนัก** ครั้งเดียว แล้วคูณกับข้อมูลทุกเดือนพร้อมกัน: $\mathbf{P}_{ตำบล\times เดือน}=\mathbf{W}\,\mathbf{X}_{เซลล์\times เดือน}$
(เซลล์ที่ไม่มีค่าในเดือนใด จะถูกตัดออกและปรับน้ำหนักที่เหลือให้รวมเป็น 1)

In [ ]:
NLAT, NLON = len(LAT), len(LON)
lat0, lon0 = LAT[0] - RES / 2, LON[0] - RES / 2
r0 = max(int(np.floor((PROV_BOUNDS[1] - lat0) / RES)) - 1, 0); r1 = min(int(np.ceil((PROV_BOUNDS[3] - lat0) / RES)) + 1, NLAT)
c0 = max(int(np.floor((PROV_BOUNDS[0] - lon0) / RES)) - 1, 0); c1 = min(int(np.ceil((PROV_BOUNDS[2] - lon0) / RES)) + 1, NLON)
rr, cc = np.meshgrid(np.arange(r0, r1), np.arange(c0, c1), indexing="ij"); rr, cc = rr.ravel(), cc.ravel()
CELLS = gpd.GeoDataFrame({"cid": rr * NLON + cc},
                         geometry=shapely.box(lon0 + cc * RES, lat0 + rr * RES, lon0 + (cc + 1) * RES, lat0 + (rr + 1) * RES), crs="EPSG:4326")
pieces = gpd.overlay(CELLS, TAMBON[["tidx", "geometry"]], how="intersection", keep_geom_type=True)
pieces["area"] = pieces.to_crs(UTM).area
W_AREA = sparse.csr_matrix((pieces.area.values, (pieces.tidx.values, pieces.cid.values)), shape=(len(TAMBON), NLAT * NLON))

# น้ำหนักประชากร: จุดศูนย์กลางเซลล์ WorldPop → ตำบล (TID) และเซลล์ดาวเทียม
ii, jj = np.nonzero((TID >= 0) & (WP > 0))
wx = WP_TR.c + (jj + .5) * WP_TR.a; wy = WP_TR.f + (ii + .5) * WP_TR.e
sr = np.floor((wy - lat0) / RES).astype(int); sc = np.floor((wx - lon0) / RES).astype(int)
ok = (sr >= 0) & (sr < NLAT) & (sc >= 0) & (sc < NLON)
W_POP = sparse.csr_matrix((WP[ii, jj][ok], (TID[ii, jj][ok], (sr * NLON + sc)[ok])), shape=(len(TAMBON), NLAT * NLON))
W_POP.sum_duplicates()
print(f"เซลล์ดาวเทียมที่ซ้อนทับจังหวัด: {len(np.unique(pieces.cid))} | ชิ้นส่วนตำบล×เซลล์: {len(pieces):,}")
print(f"เซลล์ต่อตำบล: median {np.median(np.diff(W_AREA.indptr)):.0f} (min {np.diff(W_AREA.indptr).min()}) — ตำบลที่มีเพียง 1–2 เซลล์ ค่าจะเท่ากับเซลล์นั้นเกือบทั้งหมด")

X = CUBE.reshape(len(keys), -1).T                    # [เซลล์, เวลา]
def zonal(Wm, X):
    has = np.isfinite(X)
    num = Wm @ np.where(has, X, 0.0); den = Wm @ has.astype(float)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)
PM_AREA = zonal(W_AREA, X); PM_POP = zonal(W_POP, X)
LONG = pd.DataFrame({"tidx": np.repeat(TAMBON.tidx.values, len(keys)), "time": np.tile(TIME, len(TAMBON)),
                     "pm25_area": PM_AREA.ravel(), "pm25_pop": PM_POP.ravel()})
LONG = LONG.merge(TAMBON[["tidx", "a_code", "name"] if "a_code" in TAMBON else ["tidx", "name"]], on="tidx")
print(f"✓ ค่ารายตำบล {len(TAMBON)} ตำบล × {len(keys)} ช่วงเวลา | ต่างกันระหว่างถ่วงพื้นที่กับถ่วงประชากร: "
      f"median {np.nanmedian(PM_POP - PM_AREA):+.2f} µg/m³ (ประชากรอยู่ในเซลล์ที่ค่า{'สูง' if np.nanmedian(PM_POP - PM_AREA) > 0 else 'ต่ำ'}กว่า)")

## 4. ตรวจสอบกับสถานีภาคพื้น

เทียบ **ค่าเฉลี่ยรายเดือนของสถานี** (ค่ารายวันที่ผ่านเกณฑ์ ≥ 75% ของเดือน) กับ **ค่าเซลล์ดาวเทียมที่สถานีตั้งอยู่** (และค่าเฉลี่ย 3×3 เซลล์ ~3 km)

| ตัวชี้วัด | สูตร | ความหมาย |
|---|---|---|
| r, R² | สหสัมพันธ์เพียร์สัน | ขึ้นลงตามกันหรือไม่ (ส่วนใหญ่มาจากฤดูกาล) |
| Bias | $\frac{1}{n}\sum (S-O)$ | ดาวเทียมสูง/ต่ำกว่าสถานีกี่ µg/m³ |
| NMB | $\sum (S-O) / \sum O$ | bias เป็นสัดส่วน |
| NME | $\sum \lvert S-O\rvert / \sum O$ | ความคลาดเคลื่อนสัมบูรณ์เป็นสัดส่วน |
| RMSE | $\sqrt{\frac{1}{n}\sum (S-O)^2}$ | ความคลาดเคลื่อนรวม |

$S$ = ดาวเทียม, $O$ = สถานี — ใช้ทุกสถานี คพ. ในรัศมี 200 km เพื่อให้ผลไม่ขึ้นกับ 2 สถานี และแยกรายงาน 24T/25T

In [ ]:
qc_path = os.path.join(OUTDIR, QC_FILE)
QCD = pd.read_csv(qc_path, parse_dates=["date"]).pivot_table(index="date", columns="station", values="pm25_qc") if os.path.exists(qc_path) else None

def station_monthly(sid):
    s = (QCD[sid] if (QCD is not None and sid in QCD) else T[sid]).dropna()
    s = s[s > 0]
    m = s.resample("MS").agg(["mean", "count"])
    return m["mean"].where(m["count"] / m.index.days_in_month * 100 >= MIN_PCT)

ri = lambda lat: int(np.floor((lat - lat0) / RES)); ci = lambda lon: int(np.floor((lon - lon0) / RES))
rows = []
for r in META.itertuples():
    a, b = ri(r.lat), ci(r.lon)
    if not (0 <= a < NLAT and 0 <= b < NLON): continue
    obs = station_monthly(r.sid).reindex(TIME)
    cell = CUBE[:, a, b]
    nb3 = np.nanmean(CUBE[:, max(a - 1, 0):a + 2, max(b - 1, 0):b + 2], axis=(1, 2))
    rows.append(pd.DataFrame({"sid": r.sid, "time": TIME, "obs": obs.values, "sat": cell, "sat_3x3": nb3,
                              "lat": r.lat, "lon": r.lon, "dist_km": r.dist_km}))
VAL = pd.concat(rows, ignore_index=True).dropna(subset=["obs", "sat"])
VAL["season"] = tmd_season(VAL.time + pd.Timedelta(days=14)).values
print(f"คู่ข้อมูลสถานี–เดือน: {len(VAL):,} จาก {VAL.sid.nunique()} สถานี")

def metrics(g, s="sat"):
    S, O = g[s].values, g["obs"].values
    if len(S) < 6: return pd.Series({"n": len(S)})
    return pd.Series({"n": len(S), "obs_mean": O.mean(), "sat_mean": S.mean(), "r": np.corrcoef(S, O)[0, 1],
                      "R2": np.corrcoef(S, O)[0, 1] ** 2, "bias": (S - O).mean(), "NMB_%": (S - O).sum() / O.sum() * 100,
                      "NME_%": np.abs(S - O).sum() / O.sum() * 100, "RMSE": np.sqrt(((S - O) ** 2).mean()),
                      "slope_RMA": np.sign(np.corrcoef(S, O)[0, 1]) * S.std() / O.std()})

VM = []
VM.append(metrics(VAL).rename("all stations (pooled)"))
VM.append(metrics(VAL, "sat_3x3").rename("all stations (pooled), 3×3 mean"))
for ss in SEASON_ORDER:
    VM.append(metrics(VAL[VAL.season == ss]).rename(f"all stations · {ss}"))
for sid in ["24t", "25t"]:
    g = VAL[VAL.sid == sid]
    if len(g):
        VM.append(metrics(g).rename(f"{sid.upper()} · all months"))
        for ss in SEASON_ORDER:
            VM.append(metrics(g[g.season == ss]).rename(f"{sid.upper()} · {ss}"))
VMET = pd.DataFrame(VM); display(VMET.round(2))
BYST = VAL.groupby("sid").apply(metrics, include_groups=False).join(META.set_index("sid")[["name", "dist_km"]])

In [ ]:
SEAS_COL = {"Winter": "#2c7bb6", "Summer": "#d7191c", "Rainy": "#1a9641"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
ax = axes[0]
for ss in SEASON_ORDER:
    g = VAL[VAL.season == ss]; ax.scatter(g.obs, g.sat, s=8, alpha=.35, color=SEAS_COL[ss], label=ss)
mx = np.nanpercentile(VAL[["obs", "sat"]].values, 99.5); ax.plot([0, mx], [0, mx], "k--", lw=1)
p = VMET.loc["all stations (pooled)"]
ax.set_title(f"All PCD stations ≤{VALID_RADIUS_KM} km, monthly\nr = {p.r:.2f}, NMB = {p['NMB_%']:.0f}%, RMSE = {p.RMSE:.1f}")
ax.set_xlabel("station µg/m³"); ax.set_ylabel("satellite µg/m³"); ax.legend(fontsize=8)
for ax, sid in zip(axes[1:], ["24t", "25t"]):
    g = VAL[VAL.sid == sid].sort_values("time")
    if g.empty: ax.set_visible(False); continue
    ax.plot(g.time, g.obs, "-o", ms=2.5, color="k", label="station (monthly)")
    ax.plot(g.time, g.sat, "-", color="#e08214", label="satellite cell")
    m_ = VMET.loc[f"{sid.upper()} · all months"]
    ax.set_title(f"{sid.upper()}: r = {m_.r:.2f}, bias = {m_.bias:+.1f} µg/m³, NMB = {m_['NMB_%']:+.0f}%"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
display(BYST.sort_values("dist_km").round(2).head(15))

## 5. ทักษะเชิงพื้นที่: ดาวเทียมบอกได้ไหมว่า "ที่ไหนสูงกว่า"

สำหรับการวิเคราะห์ระดับตำบล (hotspot, ความสัมพันธ์กับอัตราป่วย) สิ่งสำคัญคือ **ลำดับ** ของพื้นที่ มากกว่าค่าสัมบูรณ์
จึงเทียบค่าเฉลี่ยระยะยาวของแต่ละสถานี (เดือนที่มีทั้งสองแหล่ง) ด้วย Pearson และ **Spearman** ระหว่างสถานี

**การตีความความต่างที่ 24T:** ดาวเทียมเป็นค่าเฉลี่ยของเซลล์ ~1 km² ส่วน 24T วัด "จุด" ริมถนนใกล้โรงโม่หิน ความต่างนี้จึงเป็น **ความต่างของมาตราส่วน** ไม่ใช่ความผิดพลาดทั้งหมด
**ไม่ปรับแก้ (calibrate) ทั้งจังหวัดด้วยสมการจากสถานี** เพราะจะดึงทุกตำบลเข้าหาระดับริมถนน — ถ้าพบ bias เชิงระบบ ให้ใช้ **อันดับ/ควอร์ไทล์** เป็นตัวแปรการสัมผัสคู่กัน (ตามสไลด์ Q2)

In [ ]:
LT = VAL.groupby("sid").agg(obs=("obs", "mean"), sat=("sat", "mean"), n=("obs", "size")).query("n >= 24")
LT = LT.join(META.set_index("sid")[["name", "dist_km"]])
r_p = stats.pearsonr(LT.obs, LT.sat); r_s = stats.spearmanr(LT.obs, LT.sat)
SPATIAL = pd.DataFrame([{"stations": len(LT), "pearson_r": r_p[0], "pearson_p": r_p[1], "spearman_rho": r_s[0], "spearman_p": r_s[1]}])
display(SPATIAL.round(3))
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(LT.obs, LT.sat, s=30, c="#777")
for sid in ["24t", "25t"]:
    if sid in LT.index: ax.scatter(LT.loc[sid, "obs"], LT.loc[sid, "sat"], s=90, c="#c0504d" if sid == "24t" else "#4a7fb5", zorder=3, label=sid.upper())
lim = [min(LT.min(numeric_only=True)[["obs", "sat"]]) - 2, max(LT.max(numeric_only=True)[["obs", "sat"]]) + 2]
ax.plot(lim, lim, "k--", lw=1); ax.set_xlabel("station long-term mean µg/m³"); ax.set_ylabel("satellite long-term mean µg/m³")
ax.set_title(f"Spatial skill across {len(LT)} stations: Spearman ρ = {r_s[0]:.2f}"); ax.legend()
plt.tight_layout(); plt.show()

## 6. แผนที่ PM2.5 รายตำบล

ใช้ค่า **ถ่วงประชากร** เป็นหลัก ค่าภาพรวม = ค่าเฉลี่ยของทุกเดือน 2016–2024, รายฤดูใช้ฤดูของกรมอุตุนิยมวิทยา, รายปีใช้ปีที่มีครบ 12 เดือน
ควอร์ไทล์ (Q1–Q4) ใช้เป็นตัวแปรการสัมผัสแบบอันดับใน Part 3 ซึ่งทนต่อ bias ของดาวเทียมมากกว่าค่าสัมบูรณ์

**หมายเหตุการอ่านแผนที่รายปี:** ทุกปีใช้**สเกลสีเดียวกัน** (เปอร์เซ็นไทล์ 2–98 ของทุกปีรวมกัน) และแสดงค่าเฉลี่ยในชื่อแต่ละปี เพราะช่วงสีแคบ (~5 µg/m³) ความต่างเพียง 2–3 µg/m³ จึงดูเป็นสีจาง/เข้มชัดมาก — อ่านขนาดความต่างจากตัวเลข ไม่ใช่จากสีอย่างเดียว · ชื่อปีอยู่**เหนือ**แผนที่ของปีนั้น

In [ ]:
LONG["season"] = tmd_season(LONG.time + pd.Timedelta(days=14)).values
LONG["year"] = LONG.time.dt.year
LONG["fy_be"] = LONG.time.dt.year + 543 + (LONG.time.dt.month >= 10).astype(int)     # ปีงบประมาณ (ต.ค.–ก.ย.) พ.ศ.
SUMM = TAMBON[["tidx", "name"] + (["a_code"] if "a_code" in TAMBON else []) + ["pop", "area_km2", "geometry"]].copy()
SUMM = SUMM.merge(LONG.groupby("tidx")[["pm25_pop", "pm25_area"]].mean().add_suffix("_all"), on="tidx")
for ss in SEASON_ORDER:
    SUMM = SUMM.merge(LONG[LONG.season == ss].groupby("tidx")["pm25_pop"].mean().rename(f"pm25_pop_{ss}"), on="tidx")
full = LONG.groupby("year")["time"].nunique(); full = full[full == 12].index if SAT_TEMPORAL.lower().startswith("m") else full.index
ANNUAL = LONG[LONG.year.isin(full)].pivot_table(index="tidx", columns="year", values="pm25_pop")
SUMM = SUMM.merge(ANNUAL.add_prefix("pm25_pop_"), left_on="tidx", right_index=True)
SUMM["quartile"] = pd.qcut(SUMM.pm25_pop_all, 4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])
SUMM["rank"] = SUMM.pm25_pop_all.rank(ascending=False).astype(int)
SUMM = gpd.GeoDataFrame(SUMM, geometry="geometry", crs=TAMBON.crs)
print(f"ค่าเฉลี่ยถ่วงประชากรรายตำบล: {SUMM.pm25_pop_all.min():.1f}–{SUMM.pm25_pop_all.max():.1f} µg/m³ "
      f"(ตำบลที่เกินมาตรฐานรายปี 15 µg/m³: {(SUMM.pm25_pop_all > 15).sum()} จาก {len(SUMM)})")

ST2 = META[META.sid.isin(["24t", "25t"])]
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
ext = [LON[c0] - RES / 2, LON[c1 - 1] + RES / 2, LAT[r0] - RES / 2, LAT[r1 - 1] + RES / 2]
im = axes[0].imshow(np.nanmean(CUBE, 0)[r0:r1, c0:c1], origin="lower", extent=ext, cmap="YlOrRd")
TAMBON.boundary.plot(ax=axes[0], color="k", lw=.3); plt.colorbar(im, ax=axes[0], shrink=.7, label="µg/m³")
axes[0].set_title(f"Satellite PM2.5 grid ({RES:.2f}°), mean {keys[0]}–{keys[-1]}")
SUMM.plot("pm25_pop_all", ax=axes[1], cmap="YlOrRd", legend=True, edgecolor="k", lw=.3, legend_kwds={"shrink": .7, "label": "µg/m³"})
for ax in axes:
    ax.scatter(ST2.lon, ST2.lat, marker="^", s=80, c="cyan", edgecolor="k", zorder=5)
    for r in ST2.itertuples(): ax.annotate(r.sid.upper(), (r.lon, r.lat), xytext=(4, 4), textcoords="offset points", fontweight="bold")
    ax.set_xlim(PROV_BOUNDS[0], PROV_BOUNDS[2]); ax.set_ylim(PROV_BOUNDS[1], PROV_BOUNDS[3])
axes[1].set_title("Population-weighted PM2.5 by sub-district")
plt.tight_layout(); plt.show()

In [ ]:
v0, v1 = np.nanpercentile(SUMM[[f"pm25_pop_{s}" for s in SEASON_ORDER]].values, [2, 98])
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, ss in zip(axes, SEASON_ORDER):
    SUMM.plot(f"pm25_pop_{ss}", ax=ax, cmap="YlOrRd", vmin=v0, vmax=v1, edgecolor="k", lw=.2)
    ax.set_title(f"{ss}: mean {SUMM[f'pm25_pop_{ss}'].mean():.1f} µg/m³"); ax.set_axis_off()
fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(v0, v1), cmap="YlOrRd"), ax=axes, shrink=.6, label="µg/m³")
plt.show()
yrs = [c for c in SUMM.columns if re.fullmatch(r"pm25_pop_\d{4}", c)]
if yrs:
    ncol = 5; nrow = int(np.ceil(len(yrs) / ncol))
    v0, v1 = np.nanpercentile(SUMM[yrs].values, [2, 98])
    fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 3.2 * nrow))
    for ax, c in zip(axes.ravel(), yrs):
        SUMM.plot(c, ax=ax, cmap="YlOrRd", vmin=v0, vmax=v1, edgecolor="none")
        ax.set_title(f"{c[-4:]}  (mean {SUMM[c].mean():.1f})"); ax.set_axis_off()
    for ax in axes.ravel()[len(yrs):]: ax.set_axis_off()
    fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(v0, v1), cmap="YlOrRd"), ax=axes, shrink=.6, label="µg/m³")
    fig.suptitle("Population-weighted satellite PM2.5 by year (one colour scale for all years)", y=1.0)
    fig.savefig(os.path.join(OUTDIR, "fig_sat_pm25_annual_maps.png"), dpi=200, bbox_inches="tight")
    plt.show()
    rho = SUMM[yrs].corr(method="spearman").values[np.triu_indices(len(yrs), 1)]
    print(f"ความคงที่ของลำดับตำบลข้ามปี: Spearman ρ ระหว่างคู่ปี median {np.median(rho):.2f} (min {rho.min():.2f}) "
          "— ถ้าสูง ตำบลที่สูงก็สูงทุกปี (เป็นเรื่องโครงสร้าง ไม่ใช่เหตุการณ์ปีเดียว)")

### 6.1 ความต่างระหว่างปี: ตรวจว่ามาจากข้อมูลต้นทาง โค้ด หรือของจริง

แผนที่รายปีแสดงปี **2017 และ 2019 ต่ำกว่าปีอื่นราว 3.5 µg/m³** (ค่าเฉลี่ยตำบล 24.8 เทียบกับ 28.2–28.4 ในปี 2016/2018/2020) ส่วนปี 2021–2024 ใกล้เคียงกัน (26.9–27.7)
ผลตรวจครั้งแรก (28 ต.ค. 2569):

- ไฟล์ cache ครบ 108 ไฟล์ (12 เดือน × 9 ปี) ไม่มีไฟล์ซ้ำ (md5) รูปแบบเดียวกันทุกไฟล์ (`pm25`, กริด 294 × 307) → **ไม่ใช่ปัญหาการดาวน์โหลดหรือ zonal statistics**
- ค่าเฉลี่ยกริดในกรอบสระบุรีจากไฟล์ดิบก็สลับแบบเดียวกัน (2016 = 26.3, 2017 = 23.3, 2018 = 26.5, 2019 = 23.5, 2020 = 26.2) → **รูปแบบมาจากชุดข้อมูล SatPM2.5 เอง**
- ความต่างกระจุกในฤดูแล้ง ก.พ.–เม.ย. ฤดูฝนแทบไม่ต่าง
- ลำดับตำบลคงที่ข้ามปี (Spearman ρ median 0.89) → รูปแบบเชิงพื้นที่ใช้ได้ แม้ระดับรายปีจะขึ้นลง

Cell ถัดไปทำซ้ำการตรวจจากข้อมูลในหน่วยความจำ (`CUBE`, `LONG`, `VAL`) และเทียบกับ**สถานี คพ. ชุดเดียวกันทุกปี** เพื่อดูว่าสถานีลดลงในปี 2017/2019 ด้วยหรือไม่
ถ้าสถานีไม่ลดแต่ดาวเทียมลด → ระบุเป็นข้อจำกัดของ product (ข้อจำกัดข้อ 5 ในหัวข้อ 8) ช่วงที่ใช้กับ HDC (ต.ค. 2564 เป็นต้นไป) อยู่ในช่วง 2021–2024 ที่ค่านิ่ง จึงได้รับผลน้อย

In [ ]:
# ---- 6.1 ความต่างระหว่างปี: ไฟล์ต้นทาง vs ค่ารายตำบล vs สถานี ----
import hashlib

# (1) ไฟล์ต้นทาง: ค่าเฉลี่ยกริดในกรอบจังหวัด รายปี × เดือน + ตรวจข้อมูลซ้ำ
prov_grid = np.nanmean(CUBE[:, r0:r1, c0:c1], axis=(1, 2))
GRID = pd.DataFrame({"time": TIME, "grid_mean": prov_grid})
GRID["year"], GRID["month"] = GRID.time.dt.year, GRID.time.dt.month
GRID["md5"] = [hashlib.md5(np.ascontiguousarray(CUBE[i, r0:r1, c0:c1]).tobytes()).hexdigest()[:10] for i in range(len(TIME))]
print("กริดในกรอบจังหวัด (ไฟล์ต้นทาง) — ปี × เดือน, µg/m³")
display(GRID.pivot(index="year", columns="month", values="grid_mean").round(1))
dup = GRID[GRID.duplicated("md5", keep=False)]
print(f"เดือนที่ข้อมูลกริดซ้ำกัน: {len(dup)}" + ("" if dup.empty else f" → {sorted(dup.time.dt.strftime('%Y-%m'))}"))

# (2) ค่ารายตำบล (ค่าเฉลี่ยของ 111 ตำบล) รายปี
YR = (LONG.groupby("year").agg(n_months=("time", "nunique"), tambon_area=("pm25_area", "mean"), tambon_pop=("pm25_pop", "mean"))
          .join(GRID.groupby("year").grid_mean.mean().rename("grid_province")))
dry = GRID[GRID.month.isin([2, 3, 4])].groupby("year").grid_mean.mean().rename("grid_FebApr")
wet = GRID[GRID.month.isin(range(5, 11))].groupby("year").grid_mean.mean().rename("grid_MayOct")
YR = YR.join(dry).join(wet)

# (3) สถานี คพ. — ใช้เฉพาะสถานีที่มีข้อมูลผ่านเกณฑ์ ≥ 9 เดือนในทุกปี (ชุดสถานีเดียวกันทุกปี)
V = VAL.assign(year=VAL.time.dt.year)
cov = V.groupby(["sid", "year"]).size().unstack(fill_value=0).reindex(columns=YR.index, fill_value=0)
bal = cov.index[(cov >= 9).all(axis=1)]
VB = V[V.sid.isin(bal)]
ST = VB.groupby("year").agg(station_obs=("obs", "mean"), station_sat=("sat", "mean"))
ST["sat_minus_obs"] = ST.station_sat - ST.station_obs
YR = YR.join(ST)
YR["n_stations_balanced"] = len(bal)
print(f"\nสถานีที่มีข้อมูลครบทุกปี (≥ 9 เดือน/ปี): {len(bal)} สถานี → {', '.join(s.upper() for s in bal)}")
display(YR.round(2))

# (4) กราฟ: ระดับรายปีของดาวเทียมเทียบกับสถานี
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(YR.index, YR.tambon_pop, "o-", c="#b2182b", label="Satellite, sub-district mean (pop-weighted)")
ax[0].plot(YR.index, YR.grid_province, "s--", c="#ef8a62", label="Satellite, province grid (source files)")
if len(bal):
    ax[0].plot(YR.index, YR.station_obs, "^-", c="#2166ac", label=f"PCD stations, observed ({len(bal)} st.)")
    ax[0].plot(YR.index, YR.station_sat, "v:", c="#67a9cf", label="Satellite at the same stations")
ax[0].set_ylabel("Annual mean PM2.5 (µg/m³)"); ax[0].set_title("Annual level: satellite vs stations"); ax[0].legend(fontsize=8)
ax[1].bar(YR.index - .2, YR.grid_FebApr, .4, color="#d7191c", label="Feb–Apr (dry, hot)")
ax[1].bar(YR.index + .2, YR.grid_MayOct, .4, color="#1a9641", label="May–Oct (rainy)")
ax[1].set_ylabel("Province grid mean (µg/m³)"); ax[1].set_title("Where the year-to-year gap sits"); ax[1].legend(fontsize=8)
for a in ax: a.set_xticks(YR.index); a.grid(axis="y", alpha=.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTDIR, "fig_sat_pm25_annual_check.png"), dpi=200, bbox_inches="tight"); plt.show()
YR.to_csv(os.path.join(OUTDIR, "sat_pm25_annual_check.csv"))

# (5) อ่านผลอัตโนมัติ
odd = YR.index[(YR.index <= 2020) & (YR.index % 2 == 1)]; even = YR.index[(YR.index <= 2020) & (YR.index % 2 == 0)]
d_sat = YR.loc[even, "tambon_pop"].mean() - YR.loc[odd, "tambon_pop"].mean()
print(f"\nดาวเทียม (ตำบล): ปีคู่ 2016/18/20 สูงกว่าปีคี่ 2017/19 = {d_sat:+.2f} µg/m³")
if d_sat < 1:
    print("→ ความต่างปีคู่/ปีคี่ของดาวเทียมน้อยกว่า 1 µg/m³ — ไม่มีรูปแบบสลับปีที่ต้องอธิบาย")
elif len(bal):
    d_obs = YR.loc[even, "station_obs"].mean() - YR.loc[odd, "station_obs"].mean()
    print(f"สถานี คพ. ชุดเดียวกันทุกปี: ปีคู่ − ปีคี่ = {d_obs:+.2f} µg/m³")
    if d_obs >= 0.6 * d_sat:
        print("→ สถานีลดลงในปีคี่ด้วย: ความต่างส่วนใหญ่เป็นของจริง (สภาพอากาศ/การเผา) ไม่ใช่ข้อบกพร่องของดาวเทียม")
    elif d_obs <= 0.3 * d_sat:
        print("→ สถานีแทบไม่ลด: ความต่างส่วนใหญ่มาจาก product ดาวเทียม → รายงานเป็นข้อจำกัด และใช้ควอร์ไทล์/ค่าที่ปรับด้วยสถานีในการวิเคราะห์ความไว")
    else:
        print("→ สถานีลดลงบางส่วน: ความต่างมีทั้งของจริงและจาก product → รายงานทั้งสองอย่าง")
else:
    print("ไม่มีสถานีที่ครบทุกปี — ลดเกณฑ์ ≥ 9 เดือน/ปี หรือเทียบรายสถานีจาก VAL")

## 7. ค่า PM2.5 รายตำบลสำหรับเชื่อมกับข้อมูลผู้ป่วย HDC (Part 3)

ข้อมูลผู้ป่วย HDC 11 โรค ดึงเป็น **รายเดือน** ต.ค. 2564 – ก.ย. 2569 (ปีงบประมาณ 2565–2569) จึงต้องมีค่า PM2.5 รายตำบล **รายเดือน** ในช่วงเดียวกัน

| หัวข้อ | เนื้อหา |
|---|---|
| 7.1 | ค่าเฉลี่ยรายปีงบประมาณจากดาวเทียม (เฉพาะปีงบที่มีครบ 12 เดือน: 2560–2567) |
| 7.2 | **ตาราง PM2.5 รายตำบล × รายเดือน ต.ค. 2564 – ก.ย. 2569** — เดือนที่มีดาวเทียมใช้ค่าดาวเทียม เดือนหลัง ธ.ค. 2567 ประมาณด้วยสมการที่ (15) |
| 7.3 | ทดสอบย้อนหลังสมการที่ (15) กับช่วงที่มีดาวเทียมจริง (ความน่าเชื่อถือของค่าที่ประมาณ) |
| 7.4 | สรุปตามแหล่งที่มาของค่า และค่ารายปีงบประมาณจากตารางรายเดือน |

**สมการที่ (15)** (หัวข้อ 4.4.4 และ 7.1.4 ของโครงร่าง)

$$\hat{C}_{i,m}=\frac{\bar{S}_{i,s}}{\bar{S}_{k,s}}\times C_{k,m}$$

$\bar{S}_{i,s}$ = ค่าเฉลี่ยดาวเทียม (ถ่วงประชากร) ของตำบล $i$ ในฤดู $s$ ช่วงที่มีดาวเทียม (2016–2024 = พ.ศ. 2559–2567) ·
$\bar{S}_{k,s}$ = ค่าเฉลี่ยดาวเทียมของกริดที่ตั้งสถานี $k$ (24T หรือ 25T ที่ใกล้ตำบลที่สุด) ในฤดูเดียวกัน ·
$C_{k,m}$ = ค่าเฉลี่ยรายเดือนที่สถานี $k$ วัดได้จริง (≥ 75% ของวัน) ถ้าสถานีที่ใกล้ที่สุดไม่มีค่าในเดือนนั้น ใช้อีกสถานีหนึ่งแทน

**สเกลของค่าที่ประมาณ (`EQ15_SCALE`)** — สมการที่ (15) ตามโครงร่างหารด้วยค่าดาวเทียมที่กริดสถานี แล้วคูณค่าที่สถานีวัดจริง
ผลจึงอยู่บน **สเกลของสถานี** ถ้าดาวเทียมกับสถานีต่างกันอย่างเป็นระบบ (หัวข้อ 4: 24T ดาวเทียมต่ำกว่าสถานี ~23%) ค่ารายตำบลจะ **กระโดด** ที่รอยต่อ ธ.ค. 2567 → ม.ค. 2568
ทางเลือก `"satellite"` หารด้วย **ค่าเฉลี่ยรายฤดูที่สถานีวัดได้** ($\bar{O}_{k,s}$) แทน: $\hat{C}_{i,m}=\frac{\bar{S}_{i,s}}{\bar{O}_{k,s}}\times C_{k,m}$ ค่าที่ได้อยู่บนสเกลดาวเทียมต่อเนื่องกับช่วงก่อนหน้า
(สองแบบต่างกันแค่ตัวคูณคงที่ $\bar{S}_{k,s}/\bar{O}_{k,s}$ ต่อสถานีต่อฤดู) — **ดูผลทดสอบย้อนหลังในหัวข้อ 7.3 ก่อนเลือก** ถ้าเปลี่ยนจากโครงร่าง ให้เขียนเหตุผลในบทที่ 3

> ข้อมูลดาวเทียมรุ่น V6.GL.03 สิ้นสุด ธ.ค. 2567 (2024) ช่วงที่ต้องประมาณจึงเป็น **ม.ค. 2568 – ก.ย. 2569** (โครงร่างเขียนตามรุ่น V5.GL.05.02 ซึ่งสิ้นสุด ธ.ค. 2566 — ปรับในบทที่ 3 ฉบับวิทยานิพนธ์)

### 7.1 ค่าเฉลี่ยรายปีงบประมาณจากดาวเทียม (เฉพาะปีงบที่มีครบ 12 เดือน)

In [ ]:
cnt = LONG.groupby("fy_be")["time"].nunique()
fy_ok = cnt[cnt == 12].index if SAT_TEMPORAL.lower().startswith("m") else cnt.index
FY = (LONG[LONG.fy_be.isin(fy_ok)].groupby(["tidx", "fy_be"])[["pm25_pop", "pm25_area"]].mean().reset_index()
      .merge(TAMBON[["tidx", "name"] + (["a_code"] if "a_code" in TAMBON else [])], on="tidx"))
print("ปีงบที่มีครบ 12 เดือน:", list(fy_ok))
FY.pivot_table(index="name", columns="fy_be", values="pm25_pop").round(1).head(10)

### 7.2 ตาราง PM2.5 รายตำบล × รายเดือน (ต.ค. 2564 – ก.ย. 2569)

คอลัมน์หลักสำหรับ Part 3 คือ **`pm25`** (ดาวเทียมเมื่อมี มิฉะนั้นสมการที่ 15 ตาม `EQ15_SCALE`) และ **`pm25_source`** บอกที่มาของค่า
คอลัมน์ **`pm25_eq15`** (สเกลสถานี ตามโครงร่าง) และ **`pm25_eq15_satscale`** (สเกลดาวเทียม) คือค่าจากสมการที่ (15) **ทุกเดือน** รวมเดือนที่มีดาวเทียม — ใช้วิเคราะห์ความไวด้วยวิธีเดียวกันตลอดช่วง ไม่มีรอยต่อที่ ม.ค. 2568
คอลัมน์ **`pm25_lag1`** คือค่าเดือนก่อนหน้า (ผลต่อการเจ็บป่วยอาจตามหลังการรับสัมผัส)

In [ ]:
HDC_START, HDC_END = "2021-10-01", "2026-09-01"   # ปีงบประมาณ 2565–2569 (ต.ค. 2564 – ก.ย. 2569)
REF_STATIONS = ["24t", "25t"]                    # สถานีอ้างอิงของสมการที่ (15)
EQ15_SCALE = "station"                           # "station" = ตามโครงร่าง (หารด้วยดาวเทียมที่กริดสถานี) | "satellite" = หารด้วยค่าสถานี (ต่อเนื่องกับดาวเทียม)

def station_monthly_ext(sid):
    """ค่าเฉลี่ยรายเดือนของสถานี (≥ MIN_PCT% ของวัน) — ใช้ค่าที่ผ่าน QC จาก Notebook 01 ก่อน
    แล้วต่อด้วยไฟล์ คพ. ดิบสำหรับวันที่หลังไฟล์ QC (ไฟล์ QC อาจสิ้นสุดก่อนไฟล์ คพ. ล่าสุด)"""
    s = T[sid].dropna() if (T is not None and sid in T) else pd.Series(dtype=float)
    if QCD is not None and sid in QCD:
        q = QCD[sid].dropna()
        if len(q):
            s = pd.concat([q, s[s.index > q.index.max()]])
    s = s[s > 0]
    m = s.resample("MS").agg(["mean", "count"])
    return m["mean"].where(m["count"] / m.index.days_in_month * 100 >= MIN_PCT)

def seas(t):
    """ฤดูของเดือน (ใช้วันกลางเดือน แบบเดียวกับ LONG)"""
    return tmd_season(pd.DatetimeIndex(t) + pd.Timedelta(days=14)).values

missing_st = [s_ for s_ in REF_STATIONS if s_ not in set(META.sid)]
if T is None or missing_st:
    raise RuntimeError(f"ต้องมีข้อมูลสถานี {REF_STATIONS} สำหรับสมการที่ (15) — ไม่พบ: {missing_st or 'ไฟล์ คพ.'}")
ST_REF = META.set_index("sid").loc[REF_STATIONS, ["lat", "lon"]]
ST_OBS = pd.DataFrame({sid: station_monthly_ext(sid) for sid in REF_STATIONS})
ST_OBS.index.name, ST_OBS.columns.name = "time", "st"
ST_SAT = pd.DataFrame({sid: CUBE[:, ri(r.lat), ci(r.lon)] for sid, r in ST_REF.iterrows()}, index=TIME)
SAT_W = LONG.pivot_table(index="time", columns="tidx", values="pm25_pop")       # เดือน × ตำบล (ถ่วงประชากร)
for sid in REF_STATIONS:
    last = ST_OBS[sid].last_valid_index()
    print(f"สถานี {sid.upper()}: ค่ารายเดือนถึง {last:%Y-%m} | ดาวเทียมที่กริดสถานีเฉลี่ย {np.nanmean(ST_SAT[sid]):.1f} µg/m³")

def seasonal_ratio(months, scale="station"):
    """สัดส่วนรายฤดูแยกรายสถานี
    scale="station"  : S̄_{i,s} / S̄_{k,s}  (สมการที่ 15 ตามโครงร่าง — ผลอยู่บนสเกลสถานี)
    scale="satellite": S̄_{i,s} / Ō_{k,s}  (หารด้วยค่าเฉลี่ยที่สถานีวัดได้ — ผลอยู่บนสเกลดาวเทียม)
    ใช้เฉพาะเดือนที่มีทั้งค่าตำบล ค่ากริดสถานี และค่าสถานี (ทั้งสองแบบใช้ชุดเดือนเดียวกัน)"""
    out = {}
    for sid in REF_STATIONS:
        k = ST_SAT.loc[months, sid]
        o = ST_OBS[sid].reindex(months)
        tb = SAT_W.loc[months]
        ok = (k.notna() & o.notna()).values
        ss_ = seas(months[ok])
        num = tb[ok].groupby(ss_).mean()                 # ฤดู × ตำบล
        den = (k if scale == "station" else o)[ok].groupby(ss_).mean()
        out[sid] = num.div(den, axis=0)
    return out

def eq15(P, ratio):
    """ใส่ค่าสมการที่ (15) ให้ทุกแถวของ P (ต้องมีคอลัมน์ tidx, time, season, near_st, alt_st)"""
    R_long = pd.concat({sid: r_.stack() for sid, r_ in ratio.items()}, names=["st", "season", "tidx"]).rename("ratio").reset_index()
    O_long = ST_OBS.stack().rename("st_obs").reset_index()
    res = {}
    for tag, col in [("near", "near_st"), ("alt", "alt_st")]:
        tmp = (P[["tidx", "time", "season", col]].rename(columns={col: "st"})
               .merge(R_long, on=["st", "season", "tidx"], how="left").merge(O_long, on=["time", "st"], how="left"))
        res[tag] = (tmp["ratio"] * tmp["st_obs"]).values
    return res["near"], res["alt"]

# สถานีที่ใกล้ตำบลที่สุด (จากจุดศูนย์กลางตำบล ในพิกัด UTM)
cen = TAMBON.to_crs(UTM).geometry.centroid
stp = gpd.GeoSeries(gpd.points_from_xy(ST_REF.lon, ST_REF.lat), crs="EPSG:4326").to_crs(UTM)
dmat = np.column_stack([cen.distance(p) for p in stp])                       # ตำบล × สถานี (m)
NEAR = pd.Series(np.array(REF_STATIONS)[dmat.argmin(1)], index=TAMBON.tidx.values)
ALT = pd.Series(np.array(REF_STATIONS)[dmat.argmax(1)], index=TAMBON.tidx.values)

# ตารางตำบล × เดือน (เริ่มก่อน 1 เดือนเพื่อคำนวณ lag ของเดือนแรก)
months = pd.date_range(pd.Timestamp(HDC_START) - pd.DateOffset(months=1), HDC_END, freq="MS")
P = pd.MultiIndex.from_product([TAMBON.tidx.values, months], names=["tidx", "time"]).to_frame(index=False)
P["season"] = seas(P.time)
P = P.merge(LONG[["tidx", "time", "pm25_pop", "pm25_area"]], on=["tidx", "time"], how="left")
P["near_st"] = P.tidx.map(NEAR); P["alt_st"] = P.tidx.map(ALT)
EQ = {}
for sc_ in ["station", "satellite"]:                                          # สัดส่วนจากทุกเดือนที่มีดาวเทียม (2016–2024)
    n_, a_ = eq15(P, seasonal_ratio(TIME, sc_))
    EQ[sc_] = np.where(pd.notna(n_), n_, a_)
    if sc_ == EQ15_SCALE:
        P["eq15_near"], P["eq15_alt"] = n_, a_
P["pm25_eq15"], P["pm25_eq15_satscale"] = EQ["station"], EQ["satellite"]
print("ค่าตัวคูณสเกล S̄_k,s / Ō_k,s (ดาวเทียมที่กริดสถานี ÷ ค่าสถานี):")
for sid in REF_STATIONS:
    k_, o_ = ST_SAT[sid], ST_OBS[sid].reindex(TIME); ok_ = (k_.notna() & o_.notna()).values
    print(f"   {sid.upper()}: " + ", ".join(f"{s_} {v_:.2f}" for s_, v_ in
          (k_[ok_].groupby(seas(TIME[ok_])).mean() / o_[ok_].groupby(seas(TIME[ok_])).mean()).reindex(SEASON_ORDER).items()))

has_sat = P.pm25_pop.notna().values
has_n, has_a = P.eq15_near.notna().values, P.eq15_alt.notna().values
P["pm25"] = np.where(has_sat, P.pm25_pop, np.where(has_n, P.eq15_near, P.eq15_alt))
P["pm25_source"] = np.select([has_sat, has_n, has_a], ["satellite", "eq15_nearest_station", "eq15_other_station"], "missing")
P["ref_station"] = np.select([has_sat, has_n, has_a], ["", P.near_st.str.upper(), P.alt_st.str.upper()], "")
P = P.sort_values(["tidx", "time"])
P["pm25_lag1"] = P.groupby("tidx")["pm25"].shift(1)
P = P[P.time >= HDC_START].reset_index(drop=True)

P = P.merge(TAMBON[["tidx"] + (["a_code"] if "a_code" in TAMBON else []) + ["name"]], on="tidx")
P["ym"] = P.time.dt.strftime("%Y-%m")                                         # คีย์เชื่อมกับ HDC (ค.ศ.)
P["year_be"] = P.time.dt.year + 543; P["month"] = P.time.dt.month
P["fy_be"] = P.year_be + (P.month >= 10).astype(int)                          # ปีงบประมาณ พ.ศ.
P["nearest_station"] = P.near_st.str.upper()
PANEL = P[["tidx"] + (["a_code"] if "a_code" in P else []) + ["name", "ym", "year_be", "month", "fy_be", "season",
           "pm25", "pm25_source", "ref_station", "pm25_lag1", "pm25_pop", "pm25_area", "pm25_eq15", "pm25_eq15_satscale",
           "nearest_station"]]
PANEL = PANEL.rename(columns={"pm25_pop": "pm25_sat_pop", "pm25_area": "pm25_sat_area"})

n_m = PANEL.ym.nunique()
print(f"\nใช้สเกล EQ15_SCALE = \"{EQ15_SCALE}\" สำหรับคอลัมน์หลัก pm25")
print(f"\n✓ ตาราง {PANEL.tidx.nunique()} ตำบล × {n_m} เดือน ({PANEL.ym.min()} ถึง {PANEL.ym.max()}) = {len(PANEL):,} แถว")
print("ที่มาของค่า (จำนวนตำบล-เดือน):"); print(PANEL.pm25_source.value_counts().to_string())
miss = PANEL[PANEL.pm25_source == "missing"].ym.unique()
if len(miss):
    print(f"⚠ เดือนที่ยังไม่มีค่า: {', '.join(miss)} — สถานี 24T/25T ไม่มีค่าเฉลี่ยรายเดือนที่ผ่านเกณฑ์ "
          "(อัปเดตไฟล์ คพ. ในโฟลเดอร์ PCD_PM25 แล้วรันใหม่)")
print("ตำบลที่ใช้สถานีอ้างอิง:", NEAR.str.upper().value_counts().to_dict())

### 7.3 ทดสอบย้อนหลังสมการที่ (15)

คำนวณสัดส่วนจากปี **2016–2022** เท่านั้น แล้วประมาณค่ารายตำบลรายเดือนของปี **2023–2024** ด้วยค่าสถานีจริง เทียบกับค่าดาวเทียมจริงของปีเดียวกัน
(ข้อมูลทดสอบไม่ได้ใช้ในการคำนวณสัดส่วน) — ถ้า r สูงและ bias ต่ำ ค่าที่ประมาณในช่วง ม.ค. 2568 – ก.ย. 2569 น่าเชื่อถือในระดับเดียวกัน

* **ความแม่นรายเดือน:** r, bias, MAE, RMSE ของทุกตำบล-เดือน
* **ทักษะเชิงพื้นที่:** Spearman ระหว่างค่าเฉลี่ยรายตำบล (ประมาณ vs ดาวเทียม) — ตำบลไหนสูงกว่ายังถูกต้องหรือไม่

In [ ]:
train = TIME[TIME.year <= 2022]; test = TIME[TIME.year >= 2023]
BT = pd.MultiIndex.from_product([TAMBON.tidx.values, test], names=["tidx", "time"]).to_frame(index=False)
BT["season"] = seas(BT.time); BT["near_st"] = BT.tidx.map(NEAR); BT["alt_st"] = BT.tidx.map(ALT)
for sc_ in ["station", "satellite"]:
    n_, a_ = eq15(BT, seasonal_ratio(train, sc_))
    BT[f"est_{sc_}"] = np.where(pd.notna(n_), n_, a_)
BT["est"] = BT[f"est_{EQ15_SCALE}"]
BT = BT.merge(LONG[["tidx", "time", "pm25_pop"]], on=["tidx", "time"], how="left").dropna(subset=["est", "pm25_pop"])

def bt_metrics(g):
    e, o = g.est.values, g.pm25_pop.values
    return pd.Series({"n": len(e), "r": np.corrcoef(e, o)[0, 1], "bias": (e - o).mean(),
                      "NMB_%": (e - o).sum() / o.sum() * 100, "MAE": np.abs(e - o).mean(), "RMSE": np.sqrt(((e - o) ** 2).mean())})
BTM = pd.DataFrame([bt_metrics(BT).rename("all tambon-months")] +
                   [bt_metrics(BT[BT.season == s_]).rename(f"season · {s_}") for s_ in SEASON_ORDER] +
                   [bt_metrics(BT[BT.near_st == s_]).rename(f"ref station · {s_.upper()}") for s_ in REF_STATIONS])
BTM.insert(0, "scale", EQ15_SCALE)
_oth = "satellite" if EQ15_SCALE == "station" else "station"
_b2 = BT.assign(est=BT[f"est_{_oth}"])
BTM_ALT = pd.DataFrame([bt_metrics(_b2).rename("all tambon-months")] +
                       [bt_metrics(_b2[_b2.near_st == s_]).rename(f"ref station · {s_.upper()}") for s_ in REF_STATIONS])
BTM_ALT.insert(0, "scale", _oth)
BTM = pd.concat([BTM, BTM_ALT])
mean_t = BT.groupby("tidx")[["est", "pm25_pop"]].mean()
rho_sp, p_sp = stats.spearmanr(mean_t.est, mean_t.pm25_pop)
print(f"ทดสอบ: สัดส่วนจาก {train.min():%Y-%m}–{train.max():%Y-%m} → ประมาณ {test.min():%Y-%m}–{test.max():%Y-%m} "
      f"({BT.time.nunique()} เดือน ที่สถานีมีค่า)")
display(BTM.round(2))
print("→ แบบที่ |bias| และ RMSE ต่ำกว่า ให้ค่าที่ต่อเนื่องกับดาวเทียมมากกว่า (r เท่ากันเพราะต่างกันแค่ตัวคูณคงที่ต่อสถานีต่อฤดู)")
print(f"ทักษะเชิงพื้นที่ (ค่าเฉลี่ยรายตำบล 2023–2024): Spearman ρ = {rho_sp:.2f} (p = {p_sp:.3g})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].scatter(BT.pm25_pop, BT.est, s=4, alpha=.3)
lim = [0, np.nanpercentile(np.r_[BT.pm25_pop, BT.est], 99.5)]
axes[0].plot(lim, lim, "k--", lw=1); axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel("satellite PM2.5, tambon-month (µg/m³)"); axes[0].set_ylabel("equation (15) estimate (µg/m³)")
_m0 = BTM[BTM.scale == EQ15_SCALE].loc["all tambon-months"]
axes[0].set_title(f"Back-test 2023–2024 ({EQ15_SCALE} scale): r = {_m0.r:.2f}, RMSE = {_m0.RMSE:.1f}")
pm_ = BT.groupby("time")[["pm25_pop", "est"]].mean()
axes[1].plot(pm_.index, pm_.pm25_pop, "-o", ms=3, label="satellite (tambon mean)")
axes[1].plot(pm_.index, pm_.est, "-s", ms=3, label=f"equation (15), {EQ15_SCALE} scale")
axes[1].plot(BT.groupby("time")[f"est_{_oth}"].mean(), "--", lw=1, label=f"equation (15), {_oth} scale")
axes[1].legend(); axes[1].set_ylabel("µg/m³"); axes[1].set_title("Province mean by month")
plt.tight_layout(); plt.show()

### 7.4 สรุปตามแหล่งที่มาของค่า และค่ารายปีงบประมาณ

ค่ารายปีงบประมาณจากตารางรายเดือน (`FY_PANEL`) ใช้สำหรับการวิเคราะห์ระดับเวลา "รายปีงบประมาณ" ในหัวข้อ 7.3.3
คอลัมน์ `n_months_sat` บอกว่าค่าปีงบนั้นมาจากดาวเทียมกี่เดือน (12 = ดาวเทียมทั้งหมด, 0 = สมการที่ 15 ทั้งหมด)

In [ ]:
SRC_FY = PANEL.pivot_table(index="fy_be", columns="pm25_source", values="tidx", aggfunc="count", fill_value=0)
print("จำนวนตำบล-เดือน แยกตามปีงบประมาณและที่มาของค่า"); display(SRC_FY)

FY_PANEL = (PANEL.groupby(["tidx"] + (["a_code"] if "a_code" in PANEL else []) + ["name", "fy_be"])
            .agg(pm25=("pm25", "mean"), n_months=("pm25", "count"),
                 n_months_sat=("pm25_source", lambda s_: int((s_ == "satellite").sum())),
                 pm25_eq15=("pm25_eq15", "mean")).reset_index())
display(FY_PANEL.pivot_table(index="name", columns="fy_be", values="pm25").round(1).head(10))

COL = {"satellite": "#2c7bb6", "eq15_nearest_station": "#d7191c", "eq15_other_station": "#fdae61"}
pm_ = PANEL.assign(t=pd.to_datetime(PANEL.ym)).groupby(["t", "pm25_source"])["pm25"].mean().reset_index()
fig, ax = plt.subplots(figsize=(13, 4.2))
for src_, g in pm_.groupby("pm25_source"):
    ax.plot(g.t, g.pm25, "o", ms=4, c=COL.get(src_, "gray"), label=f"tambon mean · {src_}")
for sid, c_ in zip(REF_STATIONS, ["k", "gray"]):
    o_ = ST_OBS[sid][(ST_OBS.index >= HDC_START) & (ST_OBS.index <= HDC_END)]
    ax.plot(o_.index, o_.values, "-", c=c_, lw=1, label=f"station {sid.upper()} (observed)")
ax.axvline(TIME.max() + pd.DateOffset(months=1), ls="--", c="gray")
ax.text(TIME.max() + pd.DateOffset(months=1), ax.get_ylim()[1] * .95, "  satellite data end", va="top", fontsize=8)
ax.set_ylabel("PM2.5 (µg/m³)"); ax.set_title("Monthly PM2.5 for the HDC period (FY 2565–2569)"); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

## 8. สรุปผล บันทึกไฟล์ และข้อความในบทวิธีวิจัย

| ไฟล์ใน `OUTDIR` | เนื้อหา | ใช้ต่อใน |
|---|---|---|
| `sat_pm25_tambon_monthly.csv` | ค่ารายตำบลรายเดือน (ถ่วงพื้นที่/ประชากร) | Notebook 07, Part 3 |
| `sat_pm25_tambon_summary.gpkg` / `.csv` | ภาพรวม รายฤดู รายปี ควอร์ไทล์ อันดับ + ขอบเขตตำบล | Notebook 07 (Gi*), Part 3 |
| `sat_pm25_tambon_fiscal_year.csv` | ค่าเฉลี่ยรายปีงบประมาณ (ดาวเทียมล้วน ปีงบ 2560–2567) | ภาคผนวก |
| **`pm25_tambon_monthly_fy2565_2569.csv`** | **ตำบล × เดือน ต.ค. 2564 – ก.ย. 2569: `pm25`, `pm25_source`, `pm25_lag1`, `pm25_eq15` — คีย์เชื่อม HDC = `a_code` + `ym`** | **Notebook 13–19 (Part 3)** |
| `pm25_tambon_fiscal_year_fy2565_2569.csv` | ค่ารายปีงบประมาณ 2565–2569 จากตารางรายเดือน + จำนวนเดือนที่มาจากดาวเทียม | Part 3 (ระดับเวลารายปีงบ) |
| `sat_pm25_annual_check.csv`, `fig_sat_pm25_annual_check.png`, `fig_sat_pm25_annual_maps.png` | ความต่างระหว่างปี: ไฟล์ต้นทาง ตำบล และสถานี (หัวข้อ 6.1) | บทที่ 4 / สไลด์ |
| `eq15_backtest_metrics.csv` | ผลทดสอบย้อนหลังสมการที่ (15) | บทที่ 4 / ภาคผนวก |
| `sat_validation_monthly_pairs.csv`, `sat_validation_metrics.csv`, `sat_validation_by_station.csv`, `sat_spatial_skill.csv` | ผลตรวจสอบ | ตารางบทที่ 4 / ภาคผนวก |

In [ ]:
def fmt_p(p): return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
p = VMET.loc["all stations (pooled)"]; sp_ = SPATIAL.iloc[0]
lines = [f"PM2.5 รายเดือนจากดาวเทียม (SatPM2.5 {SAT_VERSION}, {RES:.2f}°) ช่วง {keys[0]}–{keys[-1]} เมื่อเทียบกับค่าเฉลี่ยรายเดือนของสถานี คพ. "
         f"{VAL.sid.nunique()} สถานีในรัศมี {VALID_RADIUS_KM} km ({int(p.n):,} คู่) มีสหสัมพันธ์ r = {p.r:.2f} "
         f"ค่าเฉลี่ยคลาดเคลื่อน {p.bias:+.1f} µg/m³ (NMB {p['NMB_%']:+.0f}%, RMSE {p.RMSE:.1f} µg/m³) "
         + (f"และจัดอันดับสถานีตามค่าเฉลี่ยระยะยาวได้สอดคล้อง (Spearman ρ = {sp_.spearman_rho:.2f}, {fmt_p(sp_.spearman_p)})"
            if (sp_.spearman_p < 0.05 and sp_.spearman_rho > 0) else
            f"แต่การจัดอันดับสถานีตามค่าเฉลี่ยระยะยาวยังไม่สอดคล้องอย่างมีนัยสำคัญ (Spearman ρ = {sp_.spearman_rho:.2f}, {fmt_p(sp_.spearman_p)}) "
            "จึงควรใช้ค่าดาวเทียมระดับตำบลอย่างระมัดระวังและรายงานร่วมกับตัวแปรระยะห่างจากแหล่งกำเนิด")]
for sid in ["24t", "25t"]:
    k = f"{sid.upper()} · all months"
    if k in VMET.index:
        m_ = VMET.loc[k]
        lines.append(f"ที่สถานี {sid.upper()} ดาวเทียม{'ต่ำ' if m_.bias < 0 else 'สูง'}กว่าค่าตรวจวัด {abs(m_.bias):.1f} µg/m³ (NMB {m_['NMB_%']:+.0f}%) "
                     f"โดย r = {m_.r:.2f}")
lines.append(f"ค่าเฉลี่ยถ่วงประชากรรายตำบล ({len(SUMM)} ตำบล) อยู่ระหว่าง {SUMM.pm25_pop_all.min():.1f}–{SUMM.pm25_pop_all.max():.1f} µg/m³ "
             f"(มัธยฐาน {SUMM.pm25_pop_all.median():.1f}) ตำบลที่เกินมาตรฐานรายปี 15 µg/m³ มี {(SUMM.pm25_pop_all > 15).sum()} ตำบล "
             f"ฤดูหนาวสูงที่สุด (เฉลี่ย {SUMM.pm25_pop_Winter.mean():.1f} µg/m³) และฤดูฝนต่ำที่สุด ({SUMM.pm25_pop_Rainy.mean():.1f} µg/m³)")
bt = BTM[BTM.scale == EQ15_SCALE].loc["all tambon-months"]; vc = PANEL.pm25_source.value_counts()
lines.append(f"ตารางรายตำบลรายเดือนสำหรับเชื่อมกับข้อมูลผู้ป่วย HDC ({PANEL.ym.min()} ถึง {PANEL.ym.max()}, {PANEL.tidx.nunique()} ตำบล) "
             f"ใช้ค่าดาวเทียม {vc.get('satellite', 0):,} ตำบล-เดือน และประมาณด้วยสมการที่ (15) "
             f"{vc.get('eq15_nearest_station', 0) + vc.get('eq15_other_station', 0):,} ตำบล-เดือน "
             f"(ยังไม่มีค่า {vc.get('missing', 0):,} ตำบล-เดือน) การทดสอบย้อนหลังโดยคำนวณสัดส่วนจากปี 2016–2022 "
             f"แล้วประมาณปี 2023–2024 เทียบกับค่าดาวเทียมจริง (สเกล{'สถานี' if EQ15_SCALE == 'station' else 'ดาวเทียม'}) ได้ r = {bt.r:.2f} ค่าคลาดเคลื่อนเฉลี่ย {bt.bias:+.1f} µg/m³ "
             f"(NMB {bt['NMB_%']:+.0f}%) RMSE {bt.RMSE:.1f} µg/m³ และจัดอันดับค่าเฉลี่ยรายตำบลได้สอดคล้อง Spearman ρ = {rho_sp:.2f}")
REPORT = "\n\n".join(lines); print(REPORT)

save_table(LONG.assign(time=LONG.time.dt.strftime("%Y-%m")), "sat_pm25_tambon_monthly.csv")
save_table(SUMM, "sat_pm25_tambon_summary.gpkg")
save_table(SUMM.drop(columns="geometry"), "sat_pm25_tambon_summary.csv")
save_table(FY, "sat_pm25_tambon_fiscal_year.csv")
save_table(PANEL, "pm25_tambon_monthly_fy2565_2569.csv")
save_table(FY_PANEL, "pm25_tambon_fiscal_year_fy2565_2569.csv")
save_table(BTM.reset_index().rename(columns={"index": "subset"}), "eq15_backtest_metrics.csv")
save_table(VAL, "sat_validation_monthly_pairs.csv")
save_table(VMET.reset_index().rename(columns={"index": "subset"}), "sat_validation_metrics.csv")
save_table(BYST.reset_index(), "sat_validation_by_station.csv")
save_table(SPATIAL, "sat_spatial_skill.csv")
json.dump({"satellite": f"SatPM2.5 {SAT_VERSION} {SAT_PRODUCT} {SAT_TEMPORAL} ({keys[0]}–{keys[-1]})", "boundaries": BND_SRC,
           "population": POP_SRC, "stations": "PCD daily files + Notebook 01 QC (24T/25T)"},
          open(os.path.join(OUTDIR, "sat_sources.json"), "w"), ensure_ascii=False, indent=1)
open(os.path.join(OUTDIR, "report_06_th.txt"), "w", encoding="utf-8").write(REPORT)
print("\nบันทึกแล้วใน", OUTDIR)

**ตัวอย่างข้อความในบทวิธีวิจัย**

> ความเข้มข้น PM2.5 รายตำบลรายเดือน ได้จากชุดข้อมูล SatPM2.5 รุ่น V6.GL.03 ความละเอียด 0.01° (van Donkelaar et al., 2021; Shen et al., 2024)
> ซึ่งประมาณจากความลึกเชิงแสงของละอองลอยจากดาวเทียมร่วมกับแบบจำลอง GEOS-Chem และปรับเทียบกับสถานีภาคพื้นดิน สรุปเป็นค่ารายตำบลด้วยการถ่วงน้ำหนัก
> ตามพื้นที่ซ้อนทับและตามประชากร WorldPop ปี 2020 ความละเอียด 100 เมตร (ใช้ค่าถ่วงประชากรเป็นตัวแปรหลัก) ตรวจสอบความถูกต้องกับค่าเฉลี่ยรายเดือนของ
> สถานีตรวจวัดของกรมควบคุมมลพิษในรัศมี 200 กิโลเมตร ด้วยค่าสหสัมพันธ์ ค่าคลาดเคลื่อนเฉลี่ย NMB NME และ RMSE แยกตามฤดูกาล
> สำหรับเดือนมกราคม พ.ศ. 2568 – กันยายน พ.ศ. 2569 ซึ่งยังไม่มีข้อมูลดาวเทียม ประมาณค่ารายตำบลด้วยสมการที่ (15) คือสัดส่วนระหว่างค่าดาวเทียม
> รายฤดูของตำบลกับค่าดาวเทียมรายฤดูของกริดที่ตั้งสถานี 24T หรือ 25T ที่ใกล้ตำบลที่สุด (พ.ศ. 2559–2567) คูณด้วยค่าเฉลี่ยรายเดือนที่สถานีนั้นวัดได้จริง
> และทดสอบย้อนหลังโดยคำนวณสัดส่วนจากปี พ.ศ. 2559–2565 แล้วประมาณค่าปี พ.ศ. 2566–2567 เทียบกับค่าดาวเทียมจริง

**ข้อจำกัด**
1. ชุดข้อมูลสิ้นสุด ธ.ค. 2024 — ม.ค. 2568 – ก.ย. 2569 ใช้ค่าจากสมการที่ (15) ซึ่งสมมติว่ารูปแบบเชิงพื้นที่รายฤดูของปี 2559–2567 คงเดิม และขึ้นกับสถานีเพียง 2 แห่ง (ความแม่นดูจากการทดสอบย้อนหลังในหัวข้อ 7.3; วิเคราะห์ความไวโดยจำกัดเฉพาะเดือน `pm25_source == satellite` หรือใช้ `pm25_eq15` ทั้งช่วง)
2. ความละเอียด ~1 km ไม่สามารถแยกความเข้มข้นริมถนน/ริมโรงงานได้ (ใช้ร่วมกับตัวแปรระยะห่าง: Notebook 08 และ Part 2)
3. การตรวจสอบกับสถานี คพ. ไม่เป็นอิสระทั้งหมด เพราะชุดข้อมูลใช้สถานีภาคพื้นในการปรับเทียบด้วย
4. ประชากร WorldPop ปี 2020 ปีเดียว ใช้ถ่วงน้ำหนักทุกปี
5. ระดับรายปีของ SatPM2.5 สลับขึ้นลงในปี 2016–2020 (ปี 2017 และ 2019 ต่ำกว่าปีข้างเคียงราว 3.5 µg/m³ ส่วนใหญ่ในช่วง ก.พ.–เม.ย.) ซึ่งพบตั้งแต่ไฟล์ต้นทาง ไม่ได้เกิดจากการประมวลผล (หัวข้อ 6.1) ลำดับตำบลคงที่ข้ามปี (Spearman ρ median 0.89) และช่วงที่เชื่อมกับข้อมูล HDC (ต.ค. 2564 เป็นต้นไป) อยู่ในปี 2021–2024 ที่ค่าค่อนข้างคงที่ จึงใช้ควอร์ไทล์ของตำบลเป็นตัวแปรหลักและรายงานผลเทียบสถานีรายปีไว้ในภาคผนวก

**เอกสารอ้างอิง**
- van Donkelaar, A., Hammer, M. S., Bindle, L., et al. (2021). Monthly global estimates of fine particulate matter and their uncertainty. *Environmental Science & Technology, 55*(22), 15287–15300.
- Shen, S., Li, C., van Donkelaar, A., et al. (2024). Enhancing global estimation of fine particulate matter concentrations by including geophysical a priori information in deep learning. *ACS ES&T Air*. (ตรวจรุ่นเอกสารอ้างอิงล่าสุดของ V6.GL.03 ที่ https://www.satpm.org/v6-gl-03)
- WorldPop. (2020). Global high resolution population denominators project. doi:10.5258/SOTON/WP00671
- Emery, C., et al. (2017). Recommendations on statistics and benchmarks to assess photochemical model performance. *JAWMA, 67*(5), 582–598.